<a href="https://colab.research.google.com/github/SinnottKayleigh/B2B-Sales-Algos/blob/main/Prospectoro_Scraper_(Basic).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

After colleciton of webscraped data, it will need to additionally be pre-processed.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pip install fastapi

In [ ]:
pip install selenium

In [ ]:
pip install playwright

In [ ]:
pip install fake_useragent

In [ ]:
!pip install nest_asyncio fake_useragent aiohttp beautifulsoup4 pandas

In [ ]:
import asyncio
import aiohttp
from bs4 import BeautifulSoup
from typing import Dict, List
import pandas as pd
import logging
from fake_useragent import UserAgent
import time
from urllib.parse import urlparse
import re
import nest_asyncio

nest_asyncio.apply()

class ProspectoroScraper:
    def __init__(self):
        self.headers = {
            'User-Agent': UserAgent().random,
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Connection': 'keep-alive',
        }
        self.session = None
        self.delay = 2

    async def __aenter__(self):
        await self.initialize()
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await self.cleanup()

    async def initialize(self):
        if not self.session:
            self.session = aiohttp.ClientSession(headers=self.headers)

    async def cleanup(self):
        if self.session:
            await self.session.close()
            self.session = None

    async def scrape_company(self, domain: str) -> Dict:
        try:
            # Clean the domain
            domain = self.clean_domain(domain)
            url = f"https://{domain}"

            # Basic company data
            company_data = {
                'domain': domain,
                'url': url,
                'scrape_time': time.strftime('%Y-%m-%d %H:%M:%S'),
                'data_found': False
            }

            # Scrape main website
            website_data = await self.scrape_website(url)
            if website_data:
                company_data.update(website_data)
                company_data['data_found'] = True

                # Extract contact information
                contact_info = await self.extract_contact_info(website_data.get('html_content', ''))
                company_data['contact_info'] = contact_info

                # Detect technologies
                technologies = await self.detect_technologies(website_data.get('html_content', ''))
                company_data['technologies'] = technologies

            return company_data

        except Exception as e:
            logging.error(f"Error scraping {domain}: {str(e)}")
            return {'domain': domain, 'error': str(e), 'data_found': False}

    async def scrape_website(self, url: str) -> Dict:
        try:
            await asyncio.sleep(self.delay)  # Respect rate limiting
            async with self.session.get(url, timeout=30) as response:
                if response.status == 200:
                    html_content = await response.text()
                    soup = BeautifulSoup(html_content, 'html.parser')

                    return {
                        'html_content': html_content,
                        'title': self.extract_title(soup),
                        'description': self.extract_description(soup),
                        'keywords': self.extract_keywords(soup),
                        'social_links': self.extract_social_links(soup),
                        'email_addresses': self.extract_emails(html_content)
                    }
                else:
                    logging.warning(f"Failed to fetch {url}: Status {response.status}")
                    return {}
        except Exception as e:
            logging.error(f"Error scraping website {url}: {str(e)}")
            return {}

    def extract_title(self, soup: BeautifulSoup) -> str:
        title_tag = soup.find('title')
        return title_tag.text.strip() if title_tag else ''

    def extract_description(self, soup: BeautifulSoup) -> str:
        meta_desc = soup.find('meta', attrs={'name': 'description'})
        return meta_desc.get('content', '').strip() if meta_desc else ''

    def extract_keywords(self, soup: BeautifulSoup) -> List[str]:
        meta_keywords = soup.find('meta', attrs={'name': 'keywords'})
        if meta_keywords and meta_keywords.get('content'):
            return [k.strip() for k in meta_keywords['content'].split(',')]
        return []

    def extract_social_links(self, soup: BeautifulSoup) -> Dict[str, str]:
        social_patterns = {
            'linkedin': r'linkedin\.com/(?:company|in)/',
            'twitter': r'twitter\.com/',
            'facebook': r'facebook\.com/',
            'instagram': r'instagram\.com/'
        }

        social_links = {}
        for platform, pattern in social_patterns.items():
            links = soup.find_all('a', href=re.compile(pattern))
            if links:
                social_links[platform] = links[0]['href']

        return social_links

    def extract_emails(self, html_content: str) -> List[str]:
        email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
        emails = re.findall(email_pattern, html_content)
        return list(set(emails))

    async def extract_contact_info(self, html_content: str) -> Dict:
        return {
            'emails': self.extract_emails(html_content),
            'phone_numbers': self.extract_phone_numbers(html_content),
            'addresses': self.extract_addresses(html_content)
        }

    def extract_phone_numbers(self, html_content: str) -> List[str]:
        phone_patterns = [
            r'\+\d{1,3}[-.\s]?\d{1,3}[-.\s]?\d{3,4}[-.\s]?\d{3,4}',
            r'$$\d{3}$$[-.\s]?\d{3}[-.\s]?\d{4}',
            r'\d{3}[-.\s]?\d{3}[-.\s]?\d{4}'
        ]

        phone_numbers = []
        for pattern in phone_patterns:
            matches = re.findall(pattern, html_content)
            phone_numbers.extend(matches)

        return list(set(phone_numbers))

    def extract_addresses(self, html_content: str) -> List[str]:
        address_pattern = r'\d+\s+[A-Za-z0-9\s,]+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Lane|Ln|Drive|Dr)[,\s]+[A-Za-z\s]+,\s*[A-Z]{2}\s+\d{5}'
        addresses = re.findall(address_pattern, html_content)
        return list(set(addresses))

    async def detect_technologies(self, html_content: str) -> Dict[str, List[str]]:
        tech_patterns = {
            'analytics': [
                'google-analytics.com',
                'analytics.google.com',
                'segment.com',
                'mixpanel'
            ],
            'crm': [
                'salesforce',
                'hubspot',
                'zoho',
                'pipedrive'
            ],
            'marketing': [
                'marketo',
                'mailchimp',
                'sendgrid',
                'intercom'
            ],
            'frameworks': [
                'react',
                'angular',
                'vue',
                'jquery'
            ]
        }

        detected_tech = {category: [] for category in tech_patterns}

        for category, patterns in tech_patterns.items():
            for pattern in patterns:
                if pattern.lower() in html_content.lower():
                    detected_tech[category].append(pattern)

        return detected_tech

    def clean_domain(self, domain: str) -> str:
        domain = domain.lower().strip()
        domain = re.sub(r'^https?://', '', domain)
        domain = re.sub(r'^www\.', '', domain)
        domain = domain.split('/')[0]
        return domain

async def test_real_domains():
    test_domains = [
        'microsoft.com',
        'advancedclient.io',
        'eurostar.com',
        'apple.com',
        'salesforce.com',
        'scale.com',
        'veolia.co.uk'
    ]

    results = []

    async with ProspectoroScraper() as scraper:
        for domain in test_domains:
            print(f"\nScraping {domain}...")
            try:
                result = await scraper.scrape_company(domain)
                results.append(result)

                print(f"Data found: {result['data_found']}")
                if result['data_found']:
                    print(f"Title: {result.get('title', 'N/A')}")
                    print(f"Emails found: {len(result['contact_info']['emails'])}")
                    print(f"Phone numbers found: {len(result['contact_info']['phone_numbers'])}")
                    print("Technologies detected:")
                    for category, techs in result['technologies'].items():
                        if techs:
                            print(f"  {category}: {', '.join(techs)}")

            except Exception as e:
                print(f"Error processing {domain}: {str(e)}")
                results.append({
                    'domain': domain,
                    'data_found': False,
                    'error': str(e)
                })

            print("-------------------")

    df = pd.DataFrame(results)
    df.to_csv('scraping_results.csv', index=False)
    print("\nResults saved to scraping_results.csv")

    return df

async def main():
    try:
        results_df = await test_real_domains()
        print("\nScraping Analysis:")
        print(f"Total domains processed: {len(results_df)}")
        print(f"Successful scrapes: {results_df['data_found'].sum()}")
        return results_df
    except Exception as e:
        print(f"Error in main: {str(e)}")
        return None

# Run the scraper
if __name__ == "__main__":
    results_df = asyncio.run(main())


Scraping microsoft.com...
Data found: True
Title: Microsoft – AI, Cloud, Productivity, Computing, Gaming & Apps
Emails found: 0
Phone numbers found: 9
Technologies detected:
  frameworks: jquery
-------------------

Scraping advancedclient.io...
Data found: True
Title: Advanced Client - B2B Outbound Agency
Emails found: 5
Phone numbers found: 124
Technologies detected:
-------------------

Scraping eurostar.com...
Data found: True
Title: Eurostar.com: Book Europe Train Tickets and Holidays
Emails found: 7
Phone numbers found: 43
Technologies detected:
-------------------

Scraping apple.com...
Data found: True
Title: Apple
Emails found: 0
Phone numbers found: 31
Technologies detected:
-------------------

Scraping salesforce.com...
Data found: True
Title: Salesforce: The Customer Company | Salesforce US
Emails found: 0
Phone numbers found: 58
Technologies detected:
  analytics: google-analytics.com
  crm: salesforce
-------------------

Scraping scale.com...
Data found: True
Title: Ac

In [ ]:
from fastapi import FastAPI, BackgroundTasks
from selenium import webdriver
from bs4 import BeautifulSoup
import aiohttp
import asyncio
import playwright
from typing import List, Dict
import pandas as pd
from datetime import datetime
import logging

class ProspectoroScraper:
    def __init__(self):
        self.session = aiohttp.ClientSession()
        self.rate_limiter = RateLimiter()
        self.proxy_manager = ProxyManager()
        self.data_parser = DataParser()

    class CompanyScraper:
        """Scrape company information"""
        async def scrape_company_data(self, domain: str) -> Dict:
            try:
                company_data = {
                    'website_data': await self.scrape_website(domain),
                    'linkedin_data': await self.scrape_linkedin(domain),
                    'social_data': await self.scrape_social_profiles(domain),
                    'technology_stack': await self.detect_technologies(domain),
                    'employee_count': await self.get_employee_count(domain),
                    'funding_info': await self.get_funding_info(domain)
                }
                return company_data
            except Exception as e:
                logging.error(f"Error scraping {domain}: {str(e)}")
                return None

        async def scrape_website(self, domain: str) -> Dict:
            """Scrape company website data"""
            return {
                'meta_description': '',
                'keywords': [],
                'contact_info': {},
                'about_content': '',
                'products_services': [],
                'blog_posts': []
            }

    class LinkedInScraper:
        """Scrape LinkedIn data"""
        def __init__(self):
            self.browser = playwright.chromium.launch()

        async def scrape_company_page(self, company_name: str) -> Dict:
            try:
                page = await self.browser.new_page()
                company_data = {
                    'employee_count': 0,
                    'industry': '',
                    'location': '',
                    'company_size': '',
                    'founded': '',
                    'specialties': []
                }
                return company_data
            except Exception as e:
                logging.error(f"LinkedIn scraping error: {str(e)}")
                return None

    class ContactScraper:
        """Scrape contact information"""
        async def find_contacts(self, domain: str) -> List[Dict]:
            contacts = []
            try:
                # Implement various contact finding methods
                email_patterns = await self.find_email_patterns(domain)
                linkedin_contacts = await self.find_linkedin_contacts(domain)
                contacts.extend(email_patterns)
                contacts.extend(linkedin_contacts)
                return contacts
            except Exception as e:
                logging.error(f"Contact scraping error: {str(e)}")
                return []

class ProxyManager:
    """Manage proxy rotation and health"""
    def __init__(self):
        self.proxies = self.load_proxies()
        self.health_checks = {}

    def load_proxies(self) -> List[str]:
        # Load proxies from configuration
        return []

    async def get_healthy_proxy(self) -> str:
        """Get a healthy proxy"""
        for proxy in self.proxies:
            if await self.check_proxy_health(proxy):
                return proxy
        return None

class RateLimiter:
    """Manage request rates and delays"""
    def __init__(self):
        self.requests = {}
        self.limits = {
            'linkedin': {'rpm': 20, 'delay': 3},
            'website': {'rpm': 30, 'delay': 2},
            'general': {'rpm': 40, 'delay': 1}
        }

    async def wait_if_needed(self, domain: str, scrape_type: str):
        """Check and wait if rate limit is reached"""
        current_time = datetime.now()
        key = f"{domain}_{scrape_type}"

        if key in self.requests:
            time_diff = (current_time - self.requests[key]).total_seconds()
            if time_diff < self.limits[scrape_type]['delay']:
                await asyncio.sleep(self.limits[scrape_type]['delay'] - time_diff)

        self.requests[key] = current_time

class DataParser:
    """Parse and structure scraped data"""
    async def parse_company_data(self, raw_data: Dict) -> Dict:
        """Parse and structure company data"""
        try:
            return {
                'name': self.extract_company_name(raw_data),
                'description': self.extract_description(raw_data),
                'industry': self.classify_industry(raw_data),
                'size': self.estimate_company_size(raw_data),
                'technologies': self.extract_technologies(raw_data),
                'contact_info': self.extract_contact_info(raw_data)
            }
        except Exception as e:
            logging.error(f"Parsing error: {str(e)}")
            return {}

    def extract_technologies(self, raw_data: Dict) -> List[str]:
        """Extract technology stack information"""
        technologies = []
        # Implement technology detection logic
        return technologies

# FastAPI Implementation
app = FastAPI()

@app.post("/api/v1/scrape/company")
async def scrape_company(domain: str, background_tasks: BackgroundTasks):
    """Endpoint to scrape company data"""
    scraper = ProspectoroScraper()
    background_tasks.add_task(scraper.CompanyScraper().scrape_company_data, domain)
    return {"status": "scraping_started", "domain": domain}

@app.post("/api/v1/scrape/contacts")
async def scrape_contacts(domain: str, background_tasks: BackgroundTasks):
    """Endpoint to scrape contact information"""
    scraper = ProspectoroScraper()
    background_tasks.add_task(scraper.ContactScraper().find_contacts, domain)
    return {"status": "scraping_started", "domain": domain}

@app.get("/api/v1/technology-stack/{domain}")
async def get_technology_stack(domain: str):
    """Endpoint to get technology stack information"""
    scraper = ProspectoroScraper()
    tech_stack = await scraper.CompanyScraper().detect_technologies(domain)
    return {"domain": domain, "technologies": tech_stack}

# Usage Example
async def main():
    scraper = ProspectoroScraper()

    # Scrape company data
    company_data = await scraper.CompanyScraper().scrape_company_data("example.com")

    # Find contacts
    contacts = await scraper.ContactScraper().find_contacts("example.com")

    # Parse and store results
    parsed_data = await scraper.data_parser.parse_company_data(company_data)

    return {
        "company_data": parsed_data,
        "contacts": contacts
    }

Hybrid/In-House Approach

In [ ]:
class HybridScraper:
    def __init__(self):
        self.in_house_scraper = ProspectoroScraper()
        self.api_clients = {
            'clearbit': ClearbitClient(),
            'hunter': HunterClient(),
            'builtwith': BuiltWithClient()
        }

    async def gather_company_data(self, domain: str) -> Dict:
        """Combine in-house and API data"""
        results = await asyncio.gather(
            self.in_house_scraper.scrape_company_data(domain),
            self.api_clients['clearbit'].get_company(domain),
            self.api_clients['builtwith'].get_technologies(domain)
        )
        return self.merge_results(results)

API/Scraping

In [ ]:
class APIFirstScraper:
    def __init__(self):
        self.api_clients = {}
        self.fallback_scraper = ProspectoroScraper()

    async def get_company_data(self, domain: str) -> Dict:
        """Try APIs first, fallback to scraping"""
        try:
            return await self.try_api_sources(domain)
        except APIException:
            return await self.fallback_scraper.scrape_company_data(domain)

In [ ]:
class APIFirstScraper:
    def __init__(self):
        self.api_clients = {}
        self.fallback_scraper = ProspectoroScraper()

    async def get_company_data(self, domain: str) -> Dict:
        """Try APIs first, fallback to scraping"""
        try:
            return await self.try_api_sources(domain)
        except APIException:
            return await self.fallback_scraper.scrape_company_data(domain)

Test

Domains to test the webscrape

In [ ]:
test_domains = [
    'microsoft.com',
    'advancedclient.io',
    'eurostar.com',
    'apple.com',
    'salesforce.com'
    'twitter.com',
    'scale.com',
    'veolia.co.uk'
]

Safety Considerations:

In [ ]:
class ProspectoroScraper:
    def __init__(self):
        self.rate_limit = 1  # requests per second
        self.max_retries = 3
        self.timeout = 30
        self.respect_robots_txt = True

Initial Test:

In [ ]:
test_domains = [
    'microsoft.com',
    'advancedclient.io',
    'eurostar.com',
    'apple.com',
    'salesforce.com'
    'twitter.com',
    'scale.com',
    'veolia.co.uk',
    'nasuni.com',
    'forrester.com'

]

async def monitored_test():
    start_time = time.time()
    results = await run_test()
    duration = time.time() - start_time

    print(f"\nTest completed in {duration:.2f} seconds")
    print(f"Average time per domain: {duration/len(test_domains):.2f} seconds")

    return results

Analysis of results:

In [ ]:
def analyze_scraping_results(df):
    print("\nDetailed Analysis:")

    print(f"\nTotal Domains Analyzed: {len(df)}")
    print(f"Successful Scrapes: {df['data_found'].sum()}")

    if 'social_links' in df.columns:
        print("\nSocial Media Presence:")
        social_counts = df['social_links'].apply(lambda x: {
            platform: 1 for platform in x.keys() if x
        }).sum()
        for platform, count in social_counts.items():
            print(f"{platform.capitalize()}: {count} companies")

    if 'technologies' in df.columns:
        print("\nTechnology Usage:")
        tech_counts = {}
        for tech_dict in df['technologies']:
            for category, techs in tech_dict.items():
                for tech in techs:
                    tech_counts[tech] = tech_counts.get(tech, 0) + 1

        for tech, count in sorted(tech_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"{tech}: {count} companies")

analyze_scraping_results(results_df)


Detailed Analysis:

Total Domains Analyzed: 7
Successful Scrapes: 0


Enhanced Version - No Output

In [ ]:
class EnhancedProspectoroScraper:
    def __init__(self):
        self.headers = {
            'User-Agent': UserAgent().random,
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
            'Connection': 'keep-alive',
        }
        self.session = None
        self.delay = 2

        self.intent_signals = {
            'growth_indicators': [
                'hiring', 'new office', 'expansion', 'growing team',
                'series [abcdef]', 'funding', 'investment',
                'new market', 'scaling', 'growth'
            ],
            'pain_points': [
                'challenge', 'improve', 'optimize', 'reduce costs',
                'increase efficiency', 'streamline', 'automate',
                'better solution', 'looking for', 'need help with'
            ],
            'technology_changes': [
                'migration', 'upgrade', 'implement', 'new system',
                'platform change', 'digital transformation',
                'modernize', 'legacy system'
            ],
            'buying_signals': [
                'request demo', 'get quote', 'free trial',
                'pricing', 'consultation', 'contact sales',
                'book meeting', 'learn more', 'compare plans'
            ]
        }

        self.tech_patterns = {
            'analytics': [
                'google-analytics', 'segment', 'mixpanel', 'amplitude',
                'heap', 'kissmetrics', 'hotjar', 'fullstory',
                'google tag manager', 'adobe analytics'
            ],
            'crm_sales': [
                'salesforce', 'hubspot', 'zoho', 'pipedrive',
                'freshsales', 'close.io', 'copper', 'outreach',
                'salesloft', 'groove', 'reply.io'
            ],
            'marketing': [
                'marketo', 'mailchimp', 'sendgrid', 'intercom',
                'drift', 'zendesk', 'freshdesk', 'helpscout',
                'pardot', 'eloqua', 'activecampaign'
            ],
            'development': [
                'react', 'angular', 'vue', 'jquery', 'typescript',
                'nodejs', 'python', 'java', 'php', 'ruby',
                'golang', 'aws', 'azure', 'gcp'
            ],
            'security': [
                'cloudflare', 'akamai', 'imperva', 'okta',
                'auth0', 'onelogin', 'duo', 'ssl', 'https',
                'captcha', 'recaptcha'
            ],
            'productivity': [
                'slack', 'microsoft teams', 'asana', 'jira',
                'trello', 'monday.com', 'notion', 'confluence',
                'gsuite', 'office365'
            ]
        }

    async def scrape_company(self, domain: str) -> Dict:
        try:
            await self.initialize()
            domain = self.clean_domain(domain)
            url = f"https://{domain}"

            company_data = {
                'domain': domain,
                'url': url,
                'scrape_time': time.strftime('%Y-%m-%d %H:%M:%S'),
                'data_found': False
            }

            website_data = await self.scrape_website(url)
            if website_data:
                company_data.update(website_data)
                company_data['data_found'] = True

                company_data.update({
                    'contact_info': await self.extract_contact_info(website_data['html_content']),
                    'technologies': await self.detect_technologies(website_data['html_content']),
                    'intent_signals': self.extract_intent_signals(website_data['html_content']),
                    'company_insights': await self.extract_company_insights(website_data),
                    'digital_presence': await self.analyze_digital_presence(domain),
                    'content_analysis': self.analyze_content(website_data['html_content'])
                })

            return company_data

        except Exception as e:
            logging.error(f"Error scraping {domain}: {str(e)}")
            return {'domain': domain, 'error': str(e), 'data_found': False}

    def extract_intent_signals(self, html_content: str) -> Dict:
        """Extract buying intent signals from website content"""
        intent_data = {category: [] for category in self.intent_signals}

        for category, patterns in self.intent_signals.items():
            for pattern in patterns:
                if pattern.lower() in html_content.lower():
                    intent_data[category].append(pattern)

        intent_scores = {
            'growth_score': len(intent_data['growth_indicators']) * 0.25,
            'pain_point_score': len(intent_data['pain_points']) * 0.3,
            'tech_change_score': len(intent_data['technology_changes']) * 0.2,
            'buying_intent_score': len(intent_data['buying_signals']) * 0.25
        }

        intent_data['overall_intent_score'] = sum(intent_scores.values())
        intent_data['intent_scores'] = intent_scores

        return intent_data

    async def extract_company_insights(self, website_data: Dict) -> Dict:
        """Extract comprehensive company insights"""
        html_content = website_data['html_content']
        soup = BeautifulSoup(html_content, 'html.parser')

        insights = {
            'company_size_indicators': self.extract_company_size_indicators(soup),
            'industry_focus': self.extract_industry_focus(soup),
            'target_market': self.extract_target_market(soup),
            'product_services': self.extract_products_services(soup),
            'company_age': self.extract_company_age(soup),
            'locations': self.extract_locations(soup),
            'team_info': self.extract_team_info(soup)
        }

        return insights

    def extract_company_size_indicators(self, soup: BeautifulSoup) -> Dict:
        """Extract indicators of company size"""
        indicators = {
            'employee_range': None,
            'office_locations': [],
            'enterprise_terms': False,
            'smb_terms': False
        }

        text_content = soup.get_text().lower()

        enterprise_terms = ['enterprise', 'global', 'worldwide', 'fortune 500']
        indicators['enterprise_terms'] = any(term in text_content for term in enterprise_terms)

        smb_terms = ['small business', 'startup', 'local business']
        indicators['smb_terms'] = any(term in text_content for term in smb_terms)

        return indicators

    async def analyze_digital_presence(self, domain: str) -> Dict:
        """Analyze company's digital presence"""
        presence = {
            'social_profiles': await self.get_social_profiles(domain),
            'content_platforms': await self.get_content_platforms(domain),
            'news_mentions': await self.get_news_mentions(domain),
            'job_postings': await self.get_job_postings(domain)
        }

        presence['presence_score'] = self.calculate_presence_score(presence)

        return presence

    def analyze_content(self, html_content: str) -> Dict:
        """Analyze website content for various indicators"""
        return {
            'content_topics': self.extract_main_topics(html_content),
            'content_sophistication': self.analyze_content_sophistication(html_content),
            'target_audience': self.identify_target_audience(html_content),
            'value_propositions': self.extract_value_propositions(html_content),
            'competitive_advantages': self.extract_competitive_advantages(html_content)
        }

    def extract_main_topics(self, html_content: str) -> List[str]:
        """Extract main topics from website content"""
        # Implementation using NLP techniques
        # This is a placeholder for actual NLP implementation
        return []

    def analyze_content_sophistication(self, html_content: str) -> Dict:
        """Analyze the sophistication level of the content"""
        return {
            'technical_complexity': 0,
            'business_sophistication': 0,
            'content_quality': 0
        }

    def identify_target_audience(self, html_content: str) -> List[str]:
        """Identify target audience segments"""
        audience_indicators = {
            'enterprise': ['enterprise', 'large organization', 'corporation'],
            'mid_market': ['growing companies', 'mid-size', 'medium business'],
            'small_business': ['small business', 'startup', 'entrepreneur'],
            'technical': ['developer', 'engineer', 'technical', 'IT'],
            'business': ['business user', 'manager', 'executive']
        }

        identified_audiences = []
        content_lower = html_content.lower()

        for audience, indicators in audience_indicators.items():
            if any(indicator in content_lower for indicator in indicators):
                identified_audiences.append(audience)

        return identified_audiences

    def extract_value_propositions(self, html_content: str) -> List[str]:
        """Extract value propositions from content"""
        value_prop_patterns = [
            r'(?:we|our solution|our platform|our service) (?:help|enable|empower|allow) (?:you|businesses|companies) to (?:[^.]+)',
            r'(?:increase|improve|enhance|optimize|reduce) (?:[^.]+)',
            r'(?:save|cut) (?:time|money|resources) (?:[^.]+)'
        ]

        value_props = []
        for pattern in value_prop_patterns:
            matches = re.findall(pattern, html_content, re.IGNORECASE)
            value_props.extend(matches)

        return value_props

In [ ]:
import asyncio
import aiohttp
from bs4 import BeautifulSoup
from typing import Dict, List
import pandas as pd
import logging
from fake_useragent import UserAgent
import time
import re
import nest_asyncio

nest_asyncio.apply()

class ProspectoroScraper:
    def __init__(self):
        self.headers = {
            'User-Agent': UserAgent().random,
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        }
        self.session = None

    async def initialize(self):
        """Initialize the scraper session"""
        if not self.session:
            self.session = aiohttp.ClientSession(headers=self.headers)

    async def cleanup(self):
        """Cleanup the scraper session"""
        if self.session:
            await self.session.close()

    def extract_title(self, soup: BeautifulSoup) -> str:
        """Extract page title"""
        try:
            if soup.title:
                return soup.title.string.strip()
        except Exception as e:
            logging.error(f"Error extracting title: {str(e)}")
        return ''

    def extract_description(self, soup: BeautifulSoup) -> str:
        """Extract meta description"""
        try:
            meta_desc = soup.find('meta', attrs={'name': 'description'})
            if meta_desc and meta_desc.get('content'):
                return meta_desc['content'].strip()
        except Exception as e:
            logging.error(f"Error extracting description: {str(e)}")
        return ''

    def extract_emails(self, content: str) -> List[str]:
        """Extract email addresses"""
        try:
            email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
            emails = re.findall(email_pattern, content)
            return list(set(emails))
        except Exception as e:
            logging.error(f"Error extracting emails: {str(e)}")
            return []

    def extract_phone_numbers(self, content: str) -> List[str]:
        """Extract phone numbers"""
        try:
            phone_patterns = [
                r'\+\d{1,3}[-.\s]?\d{1,3}[-.\s]?\d{3,4}[-.\s]?\d{3,4}',
                r'$$\d{3}$$[-.\s]?\d{3}[-.\s]?\d{4}',
                r'\d{3}[-.\s]?\d{3}[-.\s]?\d{4}'
            ]

            phone_numbers = []
            for pattern in phone_patterns:
                matches = re.findall(pattern, content)
                phone_numbers.extend(matches)
            return list(set(phone_numbers))
        except Exception as e:
            logging.error(f"Error extracting phone numbers: {str(e)}")
            return []

    def extract_social_links(self, soup: BeautifulSoup) -> Dict[str, str]:
        """Extract social media links"""
        try:
            social_patterns = {
                'linkedin': r'linkedin\.com/(?:company|in)/',
                'twitter': r'twitter\.com/',
                'facebook': r'facebook\.com/',
                'instagram': r'instagram\.com/'
            }

            social_links = {}
            for platform, pattern in social_patterns.items():
                links = soup.find_all('a', href=re.compile(pattern))
                if links:
                    social_links[platform] = links[0]['href']
            return social_links
        except Exception as e:
            logging.error(f"Error extracting social links: {str(e)}")
            return {}

    def detect_technologies(self, content: str) -> Dict[str, List[str]]:
        """Detect technologies used on the website"""
        try:
            tech_patterns = {
                'analytics': ['google analytics', 'mixpanel', 'segment', 'amplitude'],
                'crm': ['salesforce', 'hubspot', 'zoho', 'pipedrive'],
                'marketing': ['marketo', 'mailchimp', 'sendgrid', 'intercom'],
                'development': ['react', 'angular', 'vue', 'jquery', 'wordpress']
            }

            detected_tech = {category: [] for category in tech_patterns}
            content_lower = content.lower()

            for category, technologies in tech_patterns.items():
                for tech in technologies:
                    if tech in content_lower:
                        detected_tech[category].append(tech)

            return detected_tech
        except Exception as e:
            logging.error(f"Error detecting technologies: {str(e)}")
            return {category: [] for category in ['analytics', 'crm', 'marketing', 'development']}

    def extract_company_info(self, soup: BeautifulSoup) -> Dict:
        """Extract company information"""
        try:
            text_content = soup.get_text().lower()

            size_indicators = {
                'enterprise': ['enterprise', 'global', 'fortune 500'],
                'mid_market': ['mid-market', 'growing company'],
                'startup': ['startup', 'small business']
            }

            company_size = 'unknown'
            for size, indicators in size_indicators.items():
                if any(indicator in text_content for indicator in indicators):
                    company_size = size
                    break

            return {
                'size': company_size,
                'has_about_page': bool(soup.find('a', href=re.compile(r'about', re.I))),
                'has_contact_page': bool(soup.find('a', href=re.compile(r'contact', re.I)))
            }
        except Exception as e:
            logging.error(f"Error extracting company info: {str(e)}")
            return {'size': 'unknown', 'has_about_page': False, 'has_contact_page': False}

    async def scrape_website(self, url: str) -> Dict:
        """Scrape website and return structured data"""
        try:
            async with self.session.get(url, timeout=30) as response:
                if response.status == 200:
                    html_content = await response.text()
                    soup = BeautifulSoup(html_content, 'html.parser')

                    data = {
                        'url': url,
                        'title': self.extract_title(soup),
                        'description': self.extract_description(soup),
                        'emails': self.extract_emails(html_content),
                        'phone_numbers': self.extract_phone_numbers(html_content),
                        'social_links': self.extract_social_links(soup),
                        'technologies': self.detect_technologies(html_content),
                        'company_info': self.extract_company_info(soup),
                        'success': True
                    }

                    print(f"Scraped {url}: Found {len(data['emails'])} emails and {len(data['phone_numbers'])} phone numbers")
                    return data
                else:
                    print(f"Failed to fetch {url}: Status {response.status}")
                    return self.get_empty_result(url)
        except Exception as e:
            print(f"Error scraping {url}: {str(e)}")
            return self.get_empty_result(url)

    def get_empty_result(self, url: str) -> Dict:
        """Return empty result structure"""
        return {
            'url': url,
            'title': '',
            'description': '',
            'emails': [],
            'phone_numbers': [],
            'social_links': {},
            'technologies': {
                'analytics': [],
                'crm': [],
                'marketing': [],
                'development': []
            },
            'company_info': {
                'size': 'unknown',
                'has_about_page': False,
                'has_contact_page': False
            },
            'success': False
        }

    def clean_domain(self, domain: str) -> str:
        """Clean and validate domain"""
        domain = domain.lower().strip()
        domain = re.sub(r'^https?://', '', domain)
        domain = re.sub(r'^www\.', '', domain)
        return domain.split('/')[0]

    async def scrape_domain(self, domain: str) -> Dict:
        """Scrape a domain with proper formatting"""
        clean_domain = self.clean_domain(domain)
        url = f"https://www.{clean_domain}"
        result = await self.scrape_website(url)
        result['domain'] = clean_domain
        return result

async def test_scraper():
    """Test the scraper with multiple domains"""
    test_domains = [
        'microsoft.com',
        'advancedclient.io',
        'eurostar.com',
        'apple.com',
        'salesforce.com',
        'scale.com',
        'veolia.co.uk'
    ]

    scraper = ProspectoroScraper()
    await scraper.initialize()
    results = []

    try:
        for domain in test_domains:
            print(f"\nScraping {domain}...")
            result = await scraper.scrape_domain(domain)
            results.append(result)
            print(f"Completed scraping {domain}")
            await asyncio.sleep(2)  # Rate limiting

        df = pd.DataFrame(results)
        df.to_csv('scraping_results.csv', index=False)
        print("\nResults saved to scraping_results.csv")

        print("\nScraping Summary:")
        print(f"Total domains processed: {len(df)}")
        print(f"Successful scrapes: {df['success'].sum()}")
        print(f"Total emails found: {sum(len(x) for x in df['emails'])}")
        print(f"Total phone numbers found: {sum(len(x) for x in df['phone_numbers'])}")

        return df

    finally:
        await scraper.cleanup()

async def main():
    """Main execution function"""
    try:
        results_df = await test_scraper()

        if not results_df.empty:
            print("\nDetailed Results:")
            for _, row in results_df[results_df['success']].iterrows():
                print(f"\n{row['domain']}:")
                print(f"Title: {row['title']}")
                print(f"Emails: {len(row['emails'])}")
                print(f"Phone numbers: {len(row['phone_numbers'])}")
                if row['technologies']:
                    print("Technologies:")
                    for category, techs in row['technologies'].items():
                        if techs:
                            print(f"  {category}: {', '.join(techs)}")

        return results_df
    except Exception as e:
        print(f"Error in main: {str(e)}")
        return pd.DataFrame()

# Execute the scraper
results_df = await main()


Scraping microsoft.com...
Scraped https://www.microsoft.com: Found 0 emails and 9 phone numbers
Completed scraping microsoft.com

Scraping advancedclient.io...
Scraped https://www.advancedclient.io: Found 5 emails and 124 phone numbers
Completed scraping advancedclient.io

Scraping eurostar.com...
Scraped https://www.eurostar.com: Found 7 emails and 43 phone numbers
Completed scraping eurostar.com

Scraping apple.com...
Scraped https://www.apple.com: Found 0 emails and 31 phone numbers
Completed scraping apple.com

Scraping salesforce.com...
Scraped https://www.salesforce.com: Found 0 emails and 58 phone numbers
Completed scraping salesforce.com

Scraping scale.com...
Scraped https://www.scale.com: Found 0 emails and 62 phone numbers
Completed scraping scale.com

Scraping veolia.co.uk...
Scraped https://www.veolia.co.uk: Found 0 emails and 2 phone numbers
Completed scraping veolia.co.uk

Results saved to scraping_results.csv

Scraping Summary:
Total domains processed: 7
Successful scr

In [ ]:
results_df = await main()

if not results_df.empty:
    successful_scrapes = results_df[results_df['success']]
    if not successful_scrapes.empty:
        print("\nSuccessful Scrapes Details:")
        for _, row in successful_scrapes.iterrows():
            print(f"\nDomain: {row['domain']}")
            print(f"Title: {row['title']}")
            print(f"Emails found: {len(row['emails'])}")
            print(f"Phone numbers found: {len(row['phone_numbers'])}")
            if row['technologies']:
                print("Technologies detected:")
                for category, techs in row['technologies'].items():
                    if techs:
                        print(f"  {category}: {', '.join(techs)}")


Scraping microsoft.com...
Scraped https://www.microsoft.com: Found 0 emails and 0 phone numbers
Completed scraping microsoft.com

Scraping advancedclient.io...
Scraped https://www.advancedclient.io: Found 5 emails and 124 phone numbers
Completed scraping advancedclient.io

Scraping eurostar.com...
Scraped https://www.eurostar.com: Found 7 emails and 43 phone numbers
Completed scraping eurostar.com

Scraping apple.com...
Scraped https://www.apple.com: Found 0 emails and 31 phone numbers
Completed scraping apple.com

Scraping salesforce.com...
Scraped https://www.salesforce.com: Found 0 emails and 58 phone numbers
Completed scraping salesforce.com

Scraping scale.com...
Scraped https://www.scale.com: Found 0 emails and 62 phone numbers
Completed scraping scale.com

Scraping veolia.co.uk...
Scraped https://www.veolia.co.uk: Found 0 emails and 2 phone numbers
Completed scraping veolia.co.uk

Results saved to scraping_results.csv

Scraping Summary:
Total domains processed: 7
Successful scr

Enhanced results display

In [ ]:
def display_detailed_results(results_df: pd.DataFrame):
    """Display detailed scraping results for each website"""
    if results_df.empty:
        print("No results to display")
        return

    for _, row in results_df.iterrows():
        print("\n" + "="*50)
        print(f"DOMAIN: {row['domain']}")
        print("="*50)

        # Basic Information
        print(f"\nTitle: {row['title']}")
        print(f"Description: {row.get('description', 'None')}")

        # Emails Found
        print("\nEmails Found:")
        if row['emails'] and len(row['emails']) > 0:
            for email in row['emails']:
                print(f"  • {email}")
        else:
            print("  No emails found")

        # Phone Numbers Found
        print("\nPhone Numbers Found:")
        if row['phone_numbers'] and len(row['phone_numbers']) > 0:
            for phone in row['phone_numbers']:
                print(f"  • {phone}")
        else:
            print("  No phone numbers found")

        # Social Media Links
        print("\nSocial Media Links:")
        if row['social_links'] and len(row['social_links']) > 0:
            for platform, link in row['social_links'].items():
                print(f"  • {platform}: {link}")
        else:
            print("  No social media links found")

        # Technologies Detected
        print("\nTechnologies Detected:")
        if row['technologies']:
            for category, techs in row['technologies'].items():
                if techs:
                    print(f"  • {category.title()}:")
                    for tech in techs:
                        print(f"    - {tech}")
        else:
            print("  No technologies detected")

        # Company Information
        print("\nCompany Information:")
        if row['company_info']:
            print(f"  • Company Size: {row['company_info'].get('size', 'Unknown')}")
            print(f"  • Has About Page: {row['company_info'].get('has_about_page', False)}")
            print(f"  • Has Contact Page: {row['company_info'].get('has_contact_page', False)}")
        else:
            print("  No company information available")

        print("\n" + "-"*50)

# After running the scraper, display the results:
async def main():
    try:
        results_df = await test_scraper()
        print("\nDisplaying Detailed Results:")
        display_detailed_results(results_df)
        return results_df
    except Exception as e:
        print(f"Error in main: {str(e)}")
        return pd.DataFrame()

# Run the scraper and display results
results_df = await main()

# You can also save the detailed results to a text file:
def save_detailed_results(results_df: pd.DataFrame, filename: str = 'detailed_results.txt'):
    """Save detailed results to a text file"""
    with open(filename, 'w', encoding='utf-8') as f:
        for _, row in results_df.iterrows():
            f.write("\n" + "="*50 + "\n")
            f.write(f"DOMAIN: {row['domain']}\n")
            f.write("="*50 + "\n")

            f.write(f"\nTitle: {row['title']}\n")
            f.write(f"Description: {row.get('description', 'None')}\n")

            f.write("\nEmails Found:\n")
            if row['emails'] and len(row['emails']) > 0:
                for email in row['emails']:
                    f.write(f"  • {email}\n")
            else:
                f.write("  No emails found\n")

            f.write("\nPhone Numbers Found:\n")
            if row['phone_numbers'] and len(row['phone_numbers']) > 0:
                for phone in row['phone_numbers']:
                    f.write(f"  • {phone}\n")
            else:
                f.write("  No phone numbers found\n")

            f.write("\nSocial Media Links:\n")
            if row['social_links'] and len(row['social_links']) > 0:
                for platform, link in row['social_links'].items():
                    f.write(f"  • {platform}: {link}\n")
            else:
                f.write("  No social media links found\n")

            f.write("\nTechnologies Detected:\n")
            if row['technologies']:
                for category, techs in row['technologies'].items():
                    if techs:
                        f.write(f"  • {category.title()}:\n")
                        for tech in techs:
                            f.write(f"    - {tech}\n")
            else:
                f.write("  No technologies detected\n")

            f.write("\nCompany Information:\n")
            if row['company_info']:
                f.write(f"  • Company Size: {row['company_info'].get('size', 'Unknown')}\n")
                f.write(f"  • Has About Page: {row['company_info'].get('has_about_page', False)}\n")
                f.write(f"  • Has Contact Page: {row['company_info'].get('has_contact_page', False)}\n")
            else:
                f.write("  No company information available\n")

            f.write("\n" + "-"*50 + "\n")

# Save the results to a file
if not results_df.empty:
    save_detailed_results(results_df)
    print("\nDetailed results have been saved to 'detailed_results.txt'")

# You can also create a summary of the findings:
def create_summary(results_df: pd.DataFrame):
    """Create a summary of the findings"""
    print("\nSCRAPING SUMMARY")
    print("="*50)

    total_domains = len(results_df)
    successful_scrapes = results_df['success'].sum()
    total_emails = sum(len(x) for x in results_df['emails'])
    total_phones = sum(len(x) for x in results_df['phone_numbers'])

    print(f"\nTotal Domains Processed: {total_domains}")
    print(f"Successful Scrapes: {successful_scrapes}")
    print(f"Success Rate: {(successful_scrapes/total_domains)*100:.2f}%")

    print(f"\nTotal Emails Found: {total_emails}")
    print(f"Total Phone Numbers Found: {total_phones}")

    print("\nEmails per Domain:")
    for _, row in results_df.iterrows():
        print(f"  • {row['domain']}: {len(row['emails'])} emails")

    print("\nMost Common Technologies:")
    tech_count = {}
    for _, row in results_df.iterrows():
        for category, techs in row['technologies'].items():
            for tech in techs:
                tech_count[tech] = tech_count.get(tech, 0) + 1

    sorted_techs = sorted(tech_count.items(), key=lambda x: x[1], reverse=True)
    for tech, count in sorted_techs[:10]:
        print(f"  • {tech}: {count} occurrences")

# Create a summary of the findings
if not results_df.empty:
    create_summary(results_df)


Scraping microsoft.com...
Scraped https://www.microsoft.com: Found 0 emails and 0 phone numbers
Completed scraping microsoft.com

Scraping advancedclient.io...
Scraped https://www.advancedclient.io: Found 5 emails and 124 phone numbers
Completed scraping advancedclient.io

Scraping eurostar.com...
Scraped https://www.eurostar.com: Found 7 emails and 43 phone numbers
Completed scraping eurostar.com

Scraping apple.com...
Scraped https://www.apple.com: Found 0 emails and 31 phone numbers
Completed scraping apple.com

Scraping salesforce.com...
Scraped https://www.salesforce.com: Found 0 emails and 58 phone numbers
Completed scraping salesforce.com

Scraping scale.com...
Scraped https://www.scale.com: Found 0 emails and 62 phone numbers
Completed scraping scale.com

Scraping veolia.co.uk...
Scraped https://www.veolia.co.uk: Found 0 emails and 2 phone numbers
Completed scraping veolia.co.uk

Results saved to scraping_results.csv

Scraping Summary:
Total domains processed: 7
Successful scr

B2B-Specific Data Extraction:
- Company size and maturity indicators
- Decision-maker identification
- Technology stack analysis
- Growth and investment signals

Lead Scoring:
- Multi-factor scoring system
- ICP fit calculation
- Buying signal analysis
- Risk assessment

Business Intelligence:
- Market position analysis
- Growth indicator detection
- Technology investment signals
- Competitive analysis

Actionable Insights:
- Engagement opportunities
- Recommended actions
- Risk factors
- Priority scoring

In [ ]:
class EnhancedB2BProspector:
    def __init__(self):
        self.headers = {
            'User-Agent': UserAgent().random,
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        }
        self.session = None
        self.business_signals = {
            'company_size_indicators': {
                'enterprise': [
                    'global presence', 'enterprise solutions', 'fortune 500',
                    'multiple offices', 'worldwide locations', 'enterprise-grade'
                ],
                'mid_market': [
                    'growing company', 'medium-sized', 'regional leader',
                    'expanding business', 'multiple locations'
                ],
                'smb': [
                    'small business', 'startup', 'local business',
                    'family owned', 'newly established'
                ]
            },
            'business_indicators': {
                'growth': [
                    'expanding', 'growing', 'hiring', 'new office',
                    'recent funding', 'series [abcdef]', 'acquisition'
                ],
                'technology_investment': [
                    'digital transformation', 'modernization', 'automation',
                    'ai implementation', 'cloud migration', 'platform upgrade'
                ],
                'market_position': [
                    'market leader', 'industry pioneer', 'award-winning',
                    'recognized by', 'leading provider', 'trusted by'
                ]
            }
        }

    async def initialize(self):
        if not self.session:
            self.session = aiohttp.ClientSession(headers=self.headers)

    async def cleanup(self):
        if self.session:
            await self.session.close()

    async def fetch_page(self, url: str) -> Tuple[Optional[BeautifulSoup], Optional[str]]:
        try:
            async with self.session.get(url, timeout=30) as response:
                if response.status == 200:
                    html_content = await response.text()
                    soup = BeautifulSoup(html_content, 'html.parser')
                    return soup, html_content
                else:
                    print(f"Failed to fetch {url}: Status {response.status}")
                    return None, None
        except Exception as e:
            print(f"Error fetching {url}: {str(e)}")
            return None, None

    def extract_company_name(self, soup: BeautifulSoup) -> str:
        try:
            # Try meta tags first
            meta_name = soup.find('meta', property='og:site_name')
            if meta_name:
                return meta_name['content']

            # Try title
            title = soup.find('title')
            if title:
                return title.text.split('|')[0].strip()

            return ''
        except Exception as e:
            logging.error(f"Error extracting company name: {str(e)}")
            return ''

    def extract_industry(self, soup: BeautifulSoup) -> str:
        try:
            text_content = soup.get_text().lower()
            industries = {
                'technology': ['software', 'technology', 'it services', 'cloud', 'digital'],
                'finance': ['financial', 'banking', 'insurance', 'fintech'],
                'healthcare': ['healthcare', 'medical', 'biotech', 'pharma'],
                'manufacturing': ['manufacturing', 'industrial', 'production'],
                'retail': ['retail', 'ecommerce', 'shopping', 'consumer']
            }

            for industry, keywords in industries.items():
                if any(keyword in text_content for keyword in keywords):
                    return industry
            return 'unknown'
        except Exception as e:
            logging.error(f"Error extracting industry: {str(e)}")
            return 'unknown'

    def extract_addresses(self, soup: BeautifulSoup) -> List[str]:
        try:
            addresses = []
            # Look for address elements
            address_elements = soup.find_all(['address', 'div'], class_=re.compile('address|location', re.I))
            for element in address_elements:
                addresses.append(element.get_text().strip())

            # Look for address patterns in text
            text_content = soup.get_text()
            address_pattern = r'\d+\s+[A-Za-z0-9\s,]+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd)[,\s]+[A-Za-z\s]+,\s*[A-Z]{2}\s+\d{5}'
            found_addresses = re.findall(address_pattern, text_content)
            addresses.extend(found_addresses)

            return list(set(addresses))
        except Exception as e:
            logging.error(f"Error extracting addresses: {str(e)}")
            return []

    def extract_market_position(self, soup: BeautifulSoup, html_content: str) -> Dict:
        try:
            text_content = html_content.lower()
            position_data = {
                'market_leadership': [],
                'competitive_advantages': [],
                'customer_segments': [],
                'geographic_presence': []
            }

            # Check for market leadership indicators
            leadership_indicators = [
                'market leader', 'industry leader', 'leading provider',
                'top provider', 'premier', 'best-in-class'
            ]
            position_data['market_leadership'] = [
                indicator for indicator in leadership_indicators
                if indicator in text_content
            ]

            return position_data
        except Exception as e:
            logging.error(f"Error extracting market position: {str(e)}")
            return {}

    def calculate_lead_score(self, data: Dict) -> Dict:
        try:
            # Calculate component scores
            company_score = self.calculate_company_score(data)
            technology_score = self.calculate_technology_score(data)
            engagement_score = self.calculate_engagement_score(data)

            # Calculate weighted total score
            total_score = (
                company_score * 0.4 +
                technology_score * 0.3 +
                engagement_score * 0.3
            )

            return {
                'total_score': total_score,
                'component_scores': {
                    'company_score': company_score,
                    'technology_score': technology_score,
                    'engagement_score': engagement_score
                },
                'recommendations': self.generate_recommendations(data, total_score)
            }
        except Exception as e:
            logging.error(f"Error calculating lead score: {str(e)}")
            return {'total_score': 0, 'component_scores': {}, 'recommendations': []}

    def calculate_company_score(self, data: Dict) -> float:
        try:
            score = 0.0
            if 'company_profile' in data:
                profile = data['company_profile']
                # Size match
                if profile.get('company_size') in ['enterprise', 'mid_market']:
                    score += 0.4
                # Industry match
                if profile.get('industry') in ['technology', 'finance', 'healthcare']:
                    score += 0.3
                # Market position
                if data.get('market_position', {}).get('market_leadership', []):
                    score += 0.3
            return min(score, 1.0)
        except Exception as e:
            logging.error(f"Error calculating company score: {str(e)}")
            return 0.0

    def calculate_technology_score(self, data: Dict) -> float:
        try:
            score = 0.0
            tech_stack = data.get('technology_stack', {})
            if tech_stack:
                # Modern technologies
                if any(tech_stack.get(cat, []) for cat in ['frontend_technologies', 'backend_technologies']):
                    score += 0.4
                # Cloud services
                if tech_stack.get('cloud_services', []):
                    score += 0.3
                # Security solutions
                if tech_stack.get('security_solutions', []):
                    score += 0.3
            return min(score, 1.0)
        except Exception as e:
            logging.error(f"Error calculating technology score: {str(e)}")
            return 0.0

    def calculate_engagement_score(self, data: Dict) -> float:
        try:
            score = 0.0
            # Contact information
            if data.get('contact_information', {}).get('emails', []):
                score += 0.3
            if data.get('contact_information', {}).get('phones', []):
                score += 0.2
            # Decision makers
            if data.get('decision_makers', []):
                score += 0.5
            return min(score, 1.0)
        except Exception as e:
            logging.error(f"Error calculating engagement score: {str(e)}")
            return 0.0

    def generate_recommendations(self, data: Dict, score: float) -> List[str]:
        try:
            recommendations = []

            if score > 0.7:
                recommendations.append("High-priority lead - Immediate engagement recommended")
            elif score > 0.4:
                recommendations.append("Qualified lead - Further investigation needed")
            else:
                recommendations.append("Low-priority lead - Monitor for changes")

            # Add specific recommendations based on data
            if not data.get('contact_information', {}).get('emails', []):
                recommendations.append("Need to identify contact information")
            if not data.get('decision_makers', []):
                recommendations.append("Need to identify decision makers")

            return recommendations
        except Exception as e:
            logging.error(f"Error generating recommendations: {str(e)}")
            return []

In [ ]:
async def analyze_company(domain: str) -> Dict:
    try:
        prospector = EnhancedB2BProspector()
        await prospector.initialize()

        url = f"https://www.{domain}"
        soup, html_content = await prospector.fetch_page(url)

        if not soup or not html_content:
            return {'success': False, 'error': 'Failed to fetch webpage'}

        # Extract data
        data = {
            'company_profile': {
                'name': prospector.extract_company_name(soup),
                'industry': prospector.extract_industry(soup)
            },
            'contact_information': {
                'addresses': prospector.extract_addresses(soup)
            },
            'market_position': prospector.extract_market_position(soup, html_content)
        }

        # Calculate lead score
        lead_score = prospector.calculate_lead_score(data)

        return {
            'success': True,
            'domain': domain,
            'data': data,
            'lead_score': lead_score
        }

    except Exception as e:
        logging.error(f"Error analyzing {domain}: {str(e)}")
        return {'success': False, 'domain': domain, 'error': str(e)}
    finally:
        await prospector.cleanup()

async def main():
    domains = [
        'microsoft.com',
        'salesforce.com',
        'apple.com',
        'nasuni.com'
        'marex.com'
        'veolia.co.uk'

    ]

    results = []
    for domain in domains:
        print(f"\nAnalyzing {domain}...")
        result = await analyze_company(domain)
        results.append(result)
        await asyncio.sleep(2)  # Rate limiting

    return results

# Run the analysis
results = await main()

# Display results
for result in results:
    print(f"\n{'='*50}")
    print(f"Domain: {result['domain']}")
    if result['success']:
        print(f"Lead Score: {result['lead_score']['total_score']:.2f}")
        print("\nComponent Scores:")
        for component, score in result['lead_score']['component_scores'].items():
            print(f"  {component}: {score:.2f}")
        print("\nRecommendations:")
        for rec in result['lead_score']['recommendations']:
            print(f"  • {rec}")
    else:
        print(f"Analysis failed: {result.get('error', 'Unknown error')}")
    print('='*50)


Analyzing microsoft.com...

Analyzing salesforce.com...

Analyzing apple.com...

Analyzing nasuni.commarex.comveolia.co.uk...
Error fetching https://www.nasuni.commarex.comveolia.co.uk: Cannot connect to host www.nasuni.commarex.comveolia.co.uk:443 ssl:default [Name or service not known]

Domain: microsoft.com
Lead Score: 0.12

Component Scores:
  company_score: 0.30
  technology_score: 0.00
  engagement_score: 0.00

Recommendations:
  • Low-priority lead - Monitor for changes
  • Need to identify contact information
  • Need to identify decision makers

Domain: salesforce.com
Lead Score: 0.24

Component Scores:
  company_score: 0.60
  technology_score: 0.00
  engagement_score: 0.00

Recommendations:
  • Low-priority lead - Monitor for changes
  • Need to identify contact information
  • Need to identify decision makers

Domain: apple.com
Lead Score: 0.12

Component Scores:
  company_score: 0.30
  technology_score: 0.00
  engagement_score: 0.00

Recommendations:
  • Low-priority lead 

KeyError: 'domain'

In [ ]:
def extract_industry(self, soup: BeautifulSoup) -> str:
    """Extract industry information"""
    try:
        industries = {
            'technology': ['software', 'technology', 'it services', 'cloud', 'digital'],
            'finance': ['financial', 'banking', 'insurance', 'fintech', 'investment'],
            'healthcare': ['healthcare', 'medical', 'biotech', 'pharma', 'health'],
            'manufacturing': ['manufacturing', 'industrial', 'production', 'factory'],
            'retail': ['retail', 'ecommerce', 'shopping', 'consumer'],
            'education': ['education', 'learning', 'training', 'academic'],
            'consulting': ['consulting', 'professional services', 'advisory']
        }

        text_content = soup.get_text().lower()

        for industry, keywords in industries.items():
            if any(keyword in text_content for keyword in keywords):
                return industry
        return 'unknown'
    except Exception as e:
        logging.error(f"Error extracting industry: {str(e)}")
        return 'unknown'

def extract_addresses(self, soup: BeautifulSoup) -> List[str]:
    """Extract physical addresses"""
    try:
        addresses = []
        address_patterns = [
            r'\d+\s+[A-Za-z0-9\s,]+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Lane|Ln|Drive|Dr)[,\s]+[A-Za-z\s]+,\s*[A-Z]{2}\s+\d{5}',
            r'\d+\s+[A-Za-z0-9\s,]+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Lane|Ln|Drive|Dr)[,\s]+[A-Za-z\s]+[,\s]+[A-Z]{2,}'
        ]

        text_content = soup.get_text()
        for pattern in address_patterns:
            found_addresses = re.findall(pattern, text_content)
            addresses.extend(found_addresses)

        return list(set(addresses))
    except Exception as e:
        logging.error(f"Error extracting addresses: {str(e)}")
        return []

def extract_market_position(self, soup: BeautifulSoup, html_content: str) -> Dict:
    """Extract market position information"""
    try:
        text_content = html_content.lower()
        position_data = {
            'market_leadership': [],
            'competitive_advantages': [],
            'customer_segments': [],
            'geographic_presence': []
        }

        # Market leadership indicators
        leadership_indicators = [
            'market leader', 'industry leader', 'leading provider',
            'top provider', 'premier', 'best-in-class'
        ]
        position_data['market_leadership'] = [
            indicator for indicator in leadership_indicators
            if indicator in text_content
        ]

        return position_data
    except Exception as e:
        logging.error(f"Error extracting market position: {str(e)}")
        return {}

def calculate_lead_score(self, data: Dict) -> Dict:
    """Calculate comprehensive lead score"""
    try:
        scores = {
            'company_fit': self.calculate_company_fit(data),
            'technology_fit': self.calculate_technology_fit(data),
            'growth_potential': self.calculate_growth_potential(data),
            'engagement_likelihood': self.calculate_engagement_likelihood(data),
            'decision_maker_presence': self.calculate_decision_maker_presence(data)
        }

        # Calculate weighted total score
        weights = {
            'company_fit': 0.3,
            'technology_fit': 0.2,
            'growth_potential': 0.2,
            'engagement_likelihood': 0.15,
            'decision_maker_presence': 0.15
        }

        total_score = sum(score * weights[factor] for factor, score in scores.items())

        return {
            'total_score': total_score,
            'component_scores': scores,
            'insights': self.generate_scoring_insights(scores),
            'recommendations': self.generate_recommendations(scores, data)
        }
    except Exception as e:
        logging.error(f"Error calculating lead score: {str(e)}")
        return {'total_score': 0, 'component_scores': {}, 'insights': [], 'recommendations': []}

def calculate_company_fit(self, data: Dict) -> float:
    """Calculate company fit score"""
    try:
        score = 0.0
        if 'company_profile' in data:
            profile = data['company_profile']
            # Size match
            if profile.get('company_size') in ['enterprise', 'mid_market']:
                score += 0.3
            # Industry match
            if profile.get('industry') in ['technology', 'finance', 'healthcare']:
                score += 0.3
            # Geographic presence
            if profile.get('locations'):
                score += 0.2
        return min(score, 1.0)
    except Exception as e:
        logging.error(f"Error calculating company fit: {str(e)}")
        return 0.0

def calculate_technology_fit(self, data: Dict) -> float:
    """Calculate technology fit score"""
    try:
        score = 0.0
        if 'technology_stack' in data:
            tech_stack = data['technology_stack']
            # Modern tech stack
            if any(tech_stack.get(category, []) for category in ['cloud_services', 'modern_frameworks']):
                score += 0.4
            # Integration capabilities
            if tech_stack.get('integration_capabilities', []):
                score += 0.3
            # Security solutions
            if tech_stack.get('security_solutions', []):
                score += 0.3
        return min(score, 1.0)
    except Exception as e:
        logging.error(f"Error calculating technology fit: {str(e)}")
        return 0.0

def calculate_growth_potential(self, data: Dict) -> float:
    """Calculate growth potential score"""
    try:
        score = 0.0
        if 'business_signals' in data:
            signals = data['business_signals']
            # Growth signals
            if signals.get('growth_signals', []):
                score += 0.4
            # Investment signals
            if signals.get('investment_signals', []):
                score += 0.3
            # Technology investment
            if signals.get('technology_investment', []):
                score += 0.3
        return min(score, 1.0)
    except Exception as e:
        logging.error(f"Error calculating growth potential: {str(e)}")
        return 0.0

def calculate_engagement_likelihood(self, data: Dict) -> float:
    """Calculate engagement likelihood score"""
    try:
        score = 0.0
        # Decision maker presence
        if data.get('decision_makers', []):
            score += 0.3
        # Contact information availability
        if 'contact_information' in data:
            if data['contact_information'].get('emails', []):
                score += 0.2
            if data['contact_information'].get('phones', []):
                score += 0.2
        # Social presence
        if data.get('social_links', []):
            score += 0.3
        return min(score, 1.0)
    except Exception as e:
        logging.error(f"Error calculating engagement likelihood: {str(e)}")
        return 0.0

def calculate_decision_maker_presence(self, data: Dict) -> float:
    """Calculate decision maker presence score"""
    try:
        score = 0.0
        decision_makers = data.get('decision_makers', [])
        if decision_makers:
            # C-level presence
            if any(dm['title'].lower().startswith('c') for dm in decision_makers):
                score += 0.4
            # Director/VP level presence
            if any(any(title in dm['title'].lower() for title in ['director', 'vp', 'head']) for dm in decision_makers):
                score += 0.3
            # Technical decision makers
            if any(any(tech in dm['title'].lower() for tech in ['tech', 'it', 'engineering']) for dm in decision_makers):
                score += 0.3
        return min(score, 1.0)
    except Exception as e:
        logging.error(f"Error calculating decision maker presence: {str(e)}")
        return 0.0

def generate_scoring_insights(self, scores: Dict) -> List[str]:
    """Generate insights based on scores"""
    try:
        insights = []
        for factor, score in scores.items():
            if score > 0.7:
                insights.append(f"Strong {factor.replace('_', ' ')} indicates high potential")
            elif score < 0.3:
                insights.append(f"Low {factor.replace('_', ' ')} might need attention")
        return insights
    except Exception as e:
        logging.error(f"Error generating scoring insights: {str(e)}")
        return []

def generate_recommendations(self, scores: Dict, data: Dict) -> List[str]:
    """Generate recommendations based on scores and data"""
    try:
        recommendations = []
        # Add recommendations based on scores and data analysis
        if scores.get('company_fit', 0) > 0.7:
            recommendations.append("High priority target - Engage senior sales team")
        if scores.get('technology_fit', 0) < 0.3:
            recommendations.append("Technical validation needed - Involve solution architect")
        return recommendations
    except Exception as e:
        logging.error(f"Error generating recommendations: {str(e)}")
        return []


In [ ]:
# Run the analysis
results = await run_analysis()

# Display results summary
print("\nAnalysis Summary:")
print(f"Total companies analyzed: {len(results)}")
successful = len([r for r in results if r.get('success', False)])
print(f"Successful analyses: {successful}")
print(f"Failed analyses: {len(results) - successful}")

# Save detailed results
save_results(results)


Analyzing microsoft.com...

Analyzing salesforce.com...

Analyzing apple.com...

Analyzing nasuni.commarex.comveolia.co.uk...
Error fetching https://www.nasuni.commarex.comveolia.co.uk: Cannot connect to host www.nasuni.commarex.comveolia.co.uk:443 ssl:default [Name or service not known]

Displaying Analysis Results:

Error in analysis: 'company'

Analysis Summary:
Total companies analyzed: 0
Successful analyses: 0
Failed analyses: 0

Results saved to b2b_analysis_results.json


Enhanced Data Extractor:

In [ ]:
async def main():
    # Initialize enhanced components
    extractor = EnhancedDataExtractor()
    decision_maker_identifier = DecisionMakerIdentifier()
    scorer = EnhancedScoring()
    prospector = EnhancedB2BProspector()

    results = []
    domains = [
        'microsoft.com',
        'salesforce.com',
        'apple.com'
    ]

    try:
        await prospector.initialize()

        for domain in domains:
            print(f"\nAnalyzing {domain}...")

            # Fetch webpage content
            url = f"https://www.{domain}"
            soup, html_content = await prospector.fetch_page(url)

            if soup and html_content:
                # Store the basic data including HTML content
                result = {
                    'success': True,
                    'domain': domain,
                    'data': {
                        'html_content': html_content,
                        'soup': soup,
                        'company_profile': {
                            'name': prospector.extract_company_name(soup),
                            'industry': prospector.extract_industry(soup)
                        },
                        'contact_information': {
                            'addresses': prospector.extract_addresses(soup)
                        },
                        'market_position': prospector.extract_market_position(soup, html_content)
                    }
                }

                # Enhance with additional data
                enhanced_data = await extractor.extract_contact_details(html_content)
                result['data']['enhanced_contact_info'] = enhanced_data

                # Identify decision makers
                decision_makers = decision_maker_identifier.identify_decision_makers(
                    html_content,
                    soup
                )
                result['data']['decision_makers'] = decision_makers

                # Calculate enhanced score
                enhanced_score = scorer.calculate_comprehensive_score(result['data'])
                result['enhanced_score'] = enhanced_score

            else:
                result = {
                    'success': False,
                    'domain': domain,
                    'error': 'Failed to fetch webpage'
                }

            results.append(result)
            print(f"Completed analysis of {domain}")
            await asyncio.sleep(2)  # Rate limiting

        return results

    finally:
        await prospector.cleanup()

def run_enhanced_analysis(results: List[Dict]):
    """Run enhanced analysis and display detailed results"""
    for result in results:
        print(f"\nDetailed Analysis for {result['domain']}")
        print("="*50)

        if result['success']:
            data = result['data']

            # Display Contact Information
            print("\nContact Information:")
            print("-"*30)
            enhanced_contact = data.get('enhanced_contact_info', {})
            print(f"Emails found: {len(enhanced_contact.get('emails', []))}")
            print(f"Phone numbers found: {len(enhanced_contact.get('phones', []))}")
            print(f"Social profiles found: {len(enhanced_contact.get('social_profiles', []))}")

            # Display Decision Makers
            print("\nDecision Makers:")
            print("-"*30)
            for dm in data.get('decision_makers', []):
                print(f"• {dm.get('name', 'N/A')} - {dm.get('title', 'N/A')}")

            # Display Enhanced Score
            print("\nEnhanced Score Analysis:")
            print("-"*30)
            enhanced_score = result.get('enhanced_score', {})
            print(f"Total Score: {enhanced_score.get('total_score', 0):.2f}")

            if 'component_scores' in enhanced_score:
                print("\nComponent Scores:")
                for component, score in enhanced_score['component_scores'].items():
                    print(f"• {component}: {score:.2f}")

            # Display Recommendations
            if 'recommendations' in enhanced_score:
                print("\nRecommendations:")
                for rec in enhanced_score['recommendations']:
                    print(f"• {rec}")

        else:
            print(f"Analysis failed: {result.get('error', 'Unknown error')}")

        print("="*50)

# Run the analysis
async def run_analysis():
    try:
        results = await main()
        print("\nGenerating Detailed Analysis...")
        run_enhanced_analysis(results)

        # Save results to file
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        with open(f'detailed_analysis_{timestamp}.json', 'w') as f:
            # Convert soup objects to string to make it JSON serializable
            serializable_results = []
            for result in results:
                if result['success']:
                    result['data']['soup'] = str(result['data']['soup'])
                serializable_results.append(result)
            json.dump(serializable_results, f, indent=2)

        print(f"\nDetailed results saved to detailed_analysis_{timestamp}.json")

        return results

    except Exception as e:
        print(f"Error in analysis: {str(e)}")
        return []

# Execute the analysis
results = await run_analysis()


Analyzing microsoft.com...
Error in analysis: 'EnhancedDataExtractor' object has no attribute 'validate_emails'


CRM Integration

In [ ]:
class B2BDataExporter:
    """Handle data export to various CRM systems"""

    def __init__(self):
        self.crm_connectors = {
            'salesforce': SalesforceConnector(),
            'hubspot': HubspotConnector(),
            'pipedrive': PipedriveConnector(),
            'zoho': ZohoConnector()
        }

    async def export_to_crm(self, data: Dict, crm_type: str, config: Dict) -> Dict:
        """Export data to specified CRM"""
        try:
            if crm_type not in self.crm_connectors:
                raise ValueError(f"Unsupported CRM type: {crm_type}")

            connector = self.crm_connectors[crm_type]
            await connector.authenticate(config)

            # Transform data according to CRM requirements
            crm_data = self.transform_for_crm(data, crm_type)

            # Export to CRM
            result = await connector.export_data(crm_data)

            return {
                'success': True,
                'crm_type': crm_type,
                'exported_data': result
            }

        except Exception as e:
            return {
                'success': False,
                'crm_type': crm_type,
                'error': str(e)
            }

    def transform_for_crm(self, data: Dict, crm_type: str) -> Dict:
        """Transform scraped data into CRM-specific format"""
        transformers = {
            'salesforce': self.transform_for_salesforce,
            'hubspot': self.transform_for_hubspot,
            'pipedrive': self.transform_for_pipedrive,
            'zoho': self.transform_for_zoho
        }

        return transformers[crm_type](data)

class CRMConnector:
    """Base class for CRM connectors"""

    async def authenticate(self, config: Dict):
        raise NotImplementedError

    async def export_data(self, data: Dict):
        raise NotImplementedError

class SalesforceConnector(CRMConnector):
    async def authenticate(self, config: Dict):
        """Authenticate with Salesforce"""
        try:
            from simple_salesforce import Salesforce
            self.sf = Salesforce(
                username=config['username'],
                password=config['password'],
                security_token=config['security_token']
            )
            return True
        except Exception as e:
            raise ConnectionError(f"Salesforce authentication failed: {str(e)}")

    async def export_data(self, data: Dict) -> Dict:
        """Export data to Salesforce"""
        try:
            # Create/update account
            account_data = {
                'Name': data['company_profile']['company_name'],
                'Industry': data['company_profile']['industry_focus'],
                'Website': data['domain'],
                'Description': data['company_profile']['description']
            }

            account_id = await self.create_or_update_account(account_data)

            # Create contacts
            contact_ids = []
            for dm in data['decision_makers']:
                contact_data = {
                    'AccountId': account_id,
                    'FirstName': dm['first_name'],
                    'LastName': dm['last_name'],
                    'Title': dm['title'],
                    'Email': dm['email']
                }
                contact_id = await self.create_contact(contact_data)
                contact_ids.append(contact_id)

            return {
                'account_id': account_id,
                'contact_ids': contact_ids
            }

        except Exception as e:
            raise ExportError(f"Salesforce export failed: {str(e)}")

class HubspotConnector(CRMConnector):
    async def authenticate(self, config: Dict):
        """Authenticate with HubSpot"""
        try:
            import hubspot
            self.client = hubspot.Client.create(access_token=config['api_key'])
            return True
        except Exception as e:
            raise ConnectionError(f"HubSpot authentication failed: {str(e)}")

    async def export_data(self, data: Dict) -> Dict:
        """Export data to HubSpot"""
        try:
            # Create/update company
            company_data = {
                "properties": {
                    "name": data['company_profile']['company_name'],
                    "industry": data['company_profile']['industry_focus'],
                    "website": data['domain'],
                    "description": data['company_profile']['description']
                }
            }

            company = self.client.crm.companies.basic_api.create(company_data)

            # Create contacts
            contact_ids = []
            for dm in data['decision_makers']:
                contact_data = {
                    "properties": {
                        "firstname": dm['first_name'],
                        "lastname": dm['last_name'],
                        "jobtitle": dm['title'],
                        "email": dm['email'],
                        "associated_company_id": company.id
                    }
                }
                contact = self.client.crm.contacts.basic_api.create(contact_data)
                contact_ids.append(contact.id)

            return {
                'company_id': company.id,
                'contact_ids': contact_ids
            }

        except Exception as e:
            raise ExportError(f"HubSpot export failed: {str(e)}")

# Usage example:
async def export_to_crm_example():
    # Initialize the prospector and exporter
    prospector = EnhancedB2BProspector()
    exporter = B2BDataExporter()

    # Configure CRM credentials
    crm_config = {
        'salesforce': {
            'username': 'your_username',
            'password': 'your_password',
            'security_token': 'your_token'
        },
        'hubspot': {
            'api_key': 'your_api_key'
        }
    }

    try:
        # Analyze company
        domain = 'example.com'
        company_data = await prospector.analyze_company(domain)

        # Export to multiple CRMs
        export_results = {}
        for crm_type, config in crm_config.items():
            result = await exporter.export_to_crm(
                data=company_data,
                crm_type=crm_type,
                config=config
            )
            export_results[crm_type] = result

        return export_results

    except Exception as e:
        print(f"Export failed: {str(e)}")
        return None

# Test the export functionality
async def test_crm_export():
    print("Testing CRM export...")
    results = await export_to_crm_example()

    if results:
        print("\nExport Results:")
        for crm_type, result in results.items():
            print(f"\n{crm_type.title()}:")
            print(f"Success: {result['success']}")
            if result['success']:
                print("Exported data:", result['exported_data'])
            else:
                print("Error:", result['error'])

await test_crm_export()

Testing CRM export...


NameError: name 'PipedriveConnector' is not defined

In [ ]:
async def extract_b2b_data(self, soup: BeautifulSoup, html_content: str) -> Dict:
    """Extract comprehensive B2B data"""
    try:
        # Extract all components
        company_profile = await self.extract_company_profile(soup)
        decision_makers = await self.extract_decision_makers(soup, html_content)
        business_context = self.analyze_business_context(soup, html_content)
        tech_stack = await self.analyze_technology_stack(html_content)
        contact_info = await self.extract_contact_information(soup, html_content)
        business_signals = self.analyze_business_signals(html_content)
        market_position = self.analyze_market_position(soup, html_content)
        growth_indicators = self.analyze_growth_indicators(html_content)

        return {
            'company_profile': company_profile,
            'decision_makers': decision_makers,
            'business_context': business_context,
            'technology_stack': tech_stack,
            'contact_information': contact_info,
            'business_signals': business_signals,
            'market_position': market_position,
            'growth_indicators': growth_indicators
        }
    except Exception as e:
        logging.error(f"Error extracting B2B data: {str(e)}")
        return self.get_empty_data_structure()

def get_empty_data_structure(self) -> Dict:
    """Return empty data structure"""
    return {
        'company_profile': {},
        'decision_makers': [],
        'business_context': {},
        'technology_stack': {},
        'contact_information': {},
        'business_signals': {},
        'market_position': {},
        'growth_indicators': {}
    }

async def extract_contact_information(self, soup: BeautifulSoup, html_content: str) -> Dict:
    """Extract contact information"""
    return {
        'emails': self.extract_emails(html_content),
        'phones': self.extract_phones(html_content),
        'addresses': self.extract_addresses(soup),
        'social_profiles': self.extract_social_profiles(soup)
    }

def analyze_market_position(self, soup: BeautifulSoup, html_content: str) -> Dict:
    """Analyze market position"""
    text_content = html_content.lower()

    position_indicators = {
        'leadership': [
            'market leader', 'industry leader', 'leading provider',
            'top provider', 'premier', 'best-in-class'
        ],
        'innovation': [
            'innovative', 'cutting-edge', 'pioneering',
            'revolutionary', 'next-generation'
        ],
        'recognition': [
            'award-winning', 'recognized by', 'acclaimed',
            'certified', 'trusted by'
        ]
    }

    market_position = {category: [] for category in position_indicators}

    for category, indicators in position_indicators.items():
        for indicator in indicators:
            if indicator in text_content:
                market_position[category].append(indicator)

    return market_position

def analyze_growth_indicators(self, html_content: str) -> Dict:
    """Analyze growth indicators"""
    text_content = html_content.lower()

    growth_patterns = {
        'expansion': [
            'expanding', 'growing', 'new office',
            'new location', 'global expansion'
        ],
        'hiring': [
            'hiring', 'job openings', 'careers',
            'join our team', 'opportunities'
        ],
        'funding': [
            'series', 'funding', 'investment',
            'venture', 'capital raised'
        ],
        'acquisition': [
            'acquired', 'merger', 'partnership',
            'strategic alliance'
        ]
    }

    indicators = {category: [] for category in growth_patterns}

    for category, patterns in growth_patterns.items():
        for pattern in patterns:
            if pattern in text_content:
                indicators[category].append(pattern)

    return indicators

def extract_phones(self, content: str) -> List[str]:
    """Extract phone numbers"""
    phone_patterns = [
        r'\+\d{1,3}[-.\s]?\d{1,3}[-.\s]?\d{3,4}[-.\s]?\d{3,4}',
        r'$$\d{3}$$[-.\s]?\d{3}[-.\s]?\d{4}',
        r'\d{3}[-.\s]?\d{3}[-.\s]?\d{4}'
    ]

    phones = []
    for pattern in phone_patterns:
        matches = re.findall(pattern, content)
        phones.extend(matches)

    return list(set(phones))

def extract_social_profiles(self, soup: BeautifulSoup) -> Dict:
    """Extract social media profiles"""
    social_patterns = {
        'linkedin': r'linkedin\.com/(?:company|in)/',
        'twitter': r'twitter\.com/',
        'facebook': r'facebook\.com/',
        'instagram': r'instagram\.com/'
    }

    profiles = {}
    for platform, pattern in social_patterns.items():
        links = soup.find_all('a', href=re.compile(pattern))
        if links:
            profiles[platform] = links[0]['href']

    return profiles

def extract_addresses(self, soup: BeautifulSoup) -> List[str]:
    """Extract physical addresses"""
    address_patterns = [
        r'\d+\s+[A-Za-z0-9\s,]+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Lane|Ln|Drive|Dr)[,\s]+[A-Za-z\s]+,\s*[A-Z]{2}\s+\d{5}'
    ]

    addresses = []
    text_content = soup.get_text()

    for pattern in address_patterns:
        matches = re.findall(pattern, text_content)
        addresses.extend(matches)

    return list(set(addresses))

In [ ]:
results = await run_analysis()

if results:
    high_potential = [
        r for r in results
        if r['success'] and r['lead_score']['total_score'] > 0.7
    ]

    if high_potential:
        print("\nHigh Potential Companies:")
        for company in high_potential:
            print(f"\n{company['company']}:")
            print(f"Score: {company['lead_score']['total_score']:.2f}")
            print("Key Signals:")
            for signal in company['insights']['buying_signals'][:3]:
                print(f"  • {signal}")

Error in analysis: EnhancedB2BProspector.extract_b2b_data() missing 1 required positional argument: 'html_content'


In [ ]:
class EnhancedB2BProspector:
    def __init__(self):
        self.headers = {
            'User-Agent': UserAgent().random,
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        }
        self.session = None
        self.business_signals = {
        }

    async def initialize(self):
        if not self.session:
            self.session = aiohttp.ClientSession(headers=self.headers)

    async def cleanup(self):
        if self.session:
            await self.session.close()

    # Add this method to the EnhancedB2BProspector class
async def extract_company_profile(self, soup: BeautifulSoup) -> Dict:
    """Extract detailed company profile"""
    try:
        return {
            'name': self.extract_company_name(soup),
            'industry': self.extract_industry(soup),
            'company_size': self.determine_company_size(soup),
            'locations': self.extract_locations(soup),
            'products_services': self.extract_products_services(soup),
            'description': self.extract_company_description(soup),
            'founded_year': self.extract_founded_year(soup),
            'employee_count': self.extract_employee_count(soup)
        }
    except Exception as e:
        logging.error(f"Error extracting company profile: {str(e)}")
        return self.get_empty_company_profile()

def extract_company_name(self, soup: BeautifulSoup) -> str:
    """Extract company name"""
    try:
        # Try meta tags first
        meta_name = soup.find('meta', property='og:site_name')
        if meta_name and meta_name.get('content'):
            return meta_name['content'].strip()

        # Try title
        title = soup.find('title')
        if title:
            return title.text.split('|')[0].strip()

        return ''
    except Exception as e:
        logging.error(f"Error extracting company name: {str(e)}")
        return ''

def extract_industry(self, soup: BeautifulSoup) -> str:
    """Extract industry information"""
    try:
        text_content = soup.get_text().lower()
        industries = {
            'technology': ['software', 'technology', 'it services', 'cloud', 'digital'],
            'finance': ['financial', 'banking', 'insurance', 'fintech'],
            'healthcare': ['healthcare', 'medical', 'biotech', 'pharma'],
            'manufacturing': ['manufacturing', 'industrial', 'production'],
            'retail': ['retail', 'ecommerce', 'shopping', 'consumer']
        }

        for industry, keywords in industries.items():
            if any(keyword in text_content for keyword in keywords):
                return industry
        return 'unknown'
    except Exception as e:
        logging.error(f"Error extracting industry: {str(e)}")
        return 'unknown'

def determine_company_size(self, soup: BeautifulSoup) -> str:
    """Determine company size"""
    try:
        text_content = soup.get_text().lower()

        enterprise_indicators = ['enterprise', 'global', 'fortune 500', 'worldwide']
        midmarket_indicators = ['mid-market', 'growing company', 'regional']
        smb_indicators = ['small business', 'startup', 'local business']

        if any(indicator in text_content for indicator in enterprise_indicators):
            return 'enterprise'
        elif any(indicator in text_content for indicator in midmarket_indicators):
            return 'mid_market'
        elif any(indicator in text_content for indicator in smb_indicators):
            return 'smb'
        return 'unknown'
    except Exception as e:
        logging.error(f"Error determining company size: {str(e)}")
        return 'unknown'

def extract_locations(self, soup: BeautifulSoup) -> List[str]:
    """Extract company locations"""
    try:
        locations = []
        # Look for address elements
        address_elements = soup.find_all(['address', 'div'],
            class_=re.compile('address|location', re.I))

        for element in address_elements:
            locations.append(element.get_text().strip())

        return list(set(locations))
    except Exception as e:
        logging.error(f"Error extracting locations: {str(e)}")
        return []

def extract_products_services(self, soup: BeautifulSoup) -> List[str]:
    """Extract products and services"""
    try:
        products = []
        product_sections = soup.find_all(['div', 'section'],
            class_=re.compile('products?|services?', re.I))

        for section in product_sections:
            products.extend([
                item.get_text().strip()
                for item in section.find_all(['h2', 'h3', 'h4', 'strong'])
            ])

        return list(set(products))
    except Exception as e:
        logging.error(f"Error extracting products/services: {str(e)}")
        return []

def extract_company_description(self, soup: BeautifulSoup) -> str:
    """Extract company description"""
    try:
        # Try meta description
        meta_desc = soup.find('meta', attrs={'name': 'description'})
        if meta_desc and meta_desc.get('content'):
            return meta_desc['content'].strip()

        # Try about section
        about_section = soup.find(['div', 'section'],
            class_=re.compile('about|company', re.I))
        if about_section:
            return about_section.get_text().strip()

        return ''
    except Exception as e:
        logging.error(f"Error extracting company description: {str(e)}")
        return ''

def extract_founded_year(self, soup: BeautifulSoup) -> Optional[int]:
    """Extract company founding year"""
    try:
        text_content = soup.get_text()
        founded_patterns = [
            r'[Ff]ounded in (\d{4})',
            r'[Ee]stablished in (\d{4})',
            r'[Ss]ince (\d{4})'
        ]

        for pattern in founded_patterns:
            match = re.search(pattern, text_content)
            if match:
                return int(match.group(1))
        return None
    except Exception as e:
        logging.error(f"Error extracting founded year: {str(e)}")
        return None

def extract_employee_count(self, soup: BeautifulSoup) -> Optional[str]:
    """Extract employee count range"""
    try:
        text_content = soup.get_text()
        employee_patterns = [
            r'(\d+,?\d*)\+?\s+employees',
            r'(\d+,?\d*)\+?\s+people',
            r'team of (\d+,?\d*)\+?'
        ]

        for pattern in employee_patterns:
            match = re.search(pattern, text_content, re.I)
            if match:
                count = int(match.group(1).replace(',', ''))
                if count > 10000:
                    return '10000+'
                elif count > 1000:
                    return '1000-10000'
                elif count > 100:
                    return '100-1000'
                else:
                    return '1-100'
        return None
    except Exception as e:
        logging.error(f"Error extracting employee count: {str(e)}")
        return None

def get_empty_company_profile(self) -> Dict:
    """Return empty company profile structure"""
    return {
        'name': '',
        'industry': 'unknown',
        'company_size': 'unknown',
        'locations': [],
        'products_services': [],
        'description': '',
        'founded_year': None,
        'employee_count': None
    }

    async def analyze_company(self, domain: str) -> Dict:
        try:
            url = f"https://www.{domain}"
            soup, html_content = await self.fetch_page(url)

            if not soup or not html_content:
                return self.get_empty_result(domain)

            data = await self.extract_b2b_data(soup, html_content)
            data['domain'] = domain
            data['url'] = url
            data['success'] = True

            return data

        except Exception as e:
            logging.error(f"Error analyzing {domain}: {str(e)}")
            return self.get_empty_result(domain)

    async def extract_b2b_data(self, soup: BeautifulSoup, html_content: str) -> Dict:
        try:
            return {
                'company_profile': self.extract_company_profile(soup),
                'contact_information': self.extract_contact_information(soup, html_content),
                'technology_stack': self.extract_technology_stack(html_content),
                'business_signals': self.extract_business_signals(html_content),
                'market_position': self.extract_market_position(soup, html_content),
                'decision_makers': self.extract_decision_makers(soup, html_content)
            }
        except Exception as e:
            logging.error(f"Error extracting B2B data: {str(e)}")
            return self.get_empty_data_structure()

    def extract_company_profile(self, soup: BeautifulSoup) -> Dict:
        try:
            return {
                'name': self.extract_company_name(soup),
                'description': self.extract_description(soup),
                'industry': self.extract_industry(soup),
                'size': self.determine_company_size(soup)
            }
        except Exception as e:
            logging.error(f"Error extracting company profile: {str(e)}")
            return {}

    def extract_contact_information(self, soup: BeautifulSoup, html_content: str) -> Dict:
        try:
            return {
                'emails': self.extract_emails(html_content),
                'phones': self.extract_phones(html_content),
                'addresses': self.extract_addresses(soup),
                'social_links': self.extract_social_links(soup)
            }
        except Exception as e:
            logging.error(f"Error extracting contact information: {str(e)}")
            return {'emails': [], 'phones': [], 'addresses': [], 'social_links': {}}

    def extract_technology_stack(self, html_content: str) -> Dict:
        try:
            return {
                'analytics': self.detect_analytics_tools(html_content),
                'crm': self.detect_crm_systems(html_content),
                'marketing': self.detect_marketing_tools(html_content),
                'development': self.detect_development_tech(html_content)
            }
        except Exception as e:
            logging.error(f"Error extracting technology stack: {str(e)}")
            return {}

    def extract_business_signals(self, html_content: str) -> Dict:
        try:
            text = html_content.lower()
            signals = {}

            for category, indicators in self.business_signals.items():
                if isinstance(indicators, dict):
                    signals[category] = {
                        subcategory: [ind for ind in inds if ind.lower() in text]
                        for subcategory, inds in indicators.items()
                    }
                else:
                    signals[category] = [ind for ind in indicators if ind.lower() in text]

            return signals
        except Exception as e:
            logging.error(f"Error extracting business signals: {str(e)}")
            return {}

    # Helper methods
    def extract_company_name(self, soup: BeautifulSoup) -> str:
        try:
            # Try meta tags first
            meta_name = soup.find('meta', property='og:site_name')
            if meta_name:
                return meta_name['content']

            # Try title
            title = soup.find('title')
            if title:
                return title.text.split('|')[0].strip()

            return ''
        except:
            return ''

    def extract_description(self, soup: BeautifulSoup) -> str:
        try:
            meta_desc = soup.find('meta', {'name': 'description'})
            if meta_desc:
                return meta_desc.get('content', '').strip()
            return ''
        except:
            return ''

    def extract_emails(self, html_content: str) -> List[str]:
        try:
            email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
            emails = re.findall(email_pattern, html_content)
            return list(set(emails))
        except:
            return []

    def extract_phones(self, html_content: str) -> List[str]:
        try:
            phone_patterns = [
                r'\+\d{1,3}[-.\s]?\d{1,3}[-.\s]?\d{3,4}[-.\s]?\d{3,4}',
                r'$$\d{3}$$[-.\s]?\d{3}[-.\s]?\d{4}',
                r'\d{3}[-.\s]?\d{3}[-.\s]?\d{4}'
            ]
            phones = []
            for pattern in phone_patterns:
                phones.extend(re.findall(pattern, html_content))
            return list(set(phones))
        except:
            return []

    def extract_social_links(self, soup: BeautifulSoup) -> Dict[str, str]:
        try:
            social_patterns = {
                'linkedin': r'linkedin\.com/(?:company|in)/',
                'twitter': r'twitter\.com/',
                'facebook': r'facebook\.com/',
                'instagram': r'instagram\.com/'
            }

            social_links = {}
            for platform, pattern in social_patterns.items():
                links = soup.find_all('a', href=re.compile(pattern))
                if links:
                    social_links[platform] = links[0]['href']
            return social_links
        except:
            return {}

    def detect_analytics_tools(self, html_content: str) -> List[str]:
        tools = ['google analytics', 'mixpanel', 'segment', 'amplitude']
        return [tool for tool in tools if tool in html_content.lower()]

    def detect_crm_systems(self, html_content: str) -> List[str]:
        systems = ['salesforce', 'hubspot', 'zoho', 'pipedrive']
        return [system for system in systems if system in html_content.lower()]

    def detect_marketing_tools(self, html_content: str) -> List[str]:
        tools = ['marketo', 'mailchimp', 'sendgrid', 'intercom']
        return [tool for tool in tools if tool in html_content.lower()]

    def detect_development_tech(self, html_content: str) -> List[str]:
        tech = ['react', 'angular', 'vue', 'node.js', 'python', 'java']
        return [t for t in tech if t in html_content.lower()]

    def get_empty_result(self, domain: str) -> Dict:
        return {
            'domain': domain,
            'success': False,
            'data': self.get_empty_data_structure()
        }

    def get_empty_data_structure(self) -> Dict:
        return {
            'company_profile': {},
            'contact_information': {'emails': [], 'phones': [], 'addresses': [], 'social_links': {}},
            'technology_stack': {},
            'business_signals': {},
            'market_position': {},
            'decision_makers': []
        }

    async def fetch_page(self, url: str) -> Tuple[Optional[BeautifulSoup], Optional[str]]:
        try:
            async with self.session.get(url, timeout=30) as response:
                if response.status == 200:
                    html_content = await response.text()
                    soup = BeautifulSoup(html_content, 'html.parser')
                    return soup, html_content
                else:
                    logging.warning(f"Failed to fetch {url}: Status {response.status}")
                    return None, None
        except Exception as e:
            logging.error(f"Error fetching {url}: {str(e)}")
            return None, None

In [ ]:
async def main():
    prospector = EnhancedB2BProspector()
    await prospector.initialize()

    try:
        results = []
        for domain in ['microsoft.com', 'salesforce.com', 'apple.com']:
            print(f"\nAnalyzing {domain}...")
            result = await prospector.analyze_company(domain)
            results.append(result)
            print(f"Completed analysis of {domain}")
            await asyncio.sleep(2)  # Rate limiting

        return results
    finally:
        await prospector.cleanup()

results = await main()

for result in results:
    print(f"\nResults for {result['domain']}:")
    if result['success']:
        data = result['data']
        print("\nContact Information:")
        contact_info = data['contact_information']
        print(f"Emails found: {len(contact_info['emails'])}")
        print(f"Phone numbers found: {len(contact_info['phones'])}")

        print("\nTechnology Stack:")
        for category, techs in data['technology_stack'].items():
            if techs:
                print(f"{category}: {', '.join(techs)}")

        print("\nBusiness Signals:")
        for category, signals in data['business_signals'].items():
            if signals:
                print(f"{category}: {signals}")
    else:
        print("Analysis failed")


Analyzing microsoft.com...


AttributeError: 'EnhancedB2BProspector' object has no attribute 'analyze_company'

In [ ]:
def extract_industry(self, soup: BeautifulSoup) -> str:
    """Extract industry information"""
    try:
        industries = {
            'technology': ['software', 'technology', 'it services', 'cloud', 'digital'],
            'finance': ['financial', 'banking', 'insurance', 'fintech', 'investment'],
            'healthcare': ['healthcare', 'medical', 'biotech', 'pharma', 'health'],
            'manufacturing': ['manufacturing', 'industrial', 'production', 'factory'],
            'retail': ['retail', 'ecommerce', 'shopping', 'consumer'],
            'education': ['education', 'learning', 'training', 'academic'],
            'consulting': ['consulting', 'professional services', 'advisory']
        }

        text_content = soup.get_text().lower()

        for industry, keywords in industries.items():
            if any(keyword in text_content for keyword in keywords):
                return industry

        return 'unknown'
    except Exception as e:
        logging.error(f"Error extracting industry: {str(e)}")
        return 'unknown'

def extract_addresses(self, soup: BeautifulSoup) -> List[str]:
    """Extract physical addresses"""
    try:
        addresses = []

        # Common address patterns
        address_patterns = [
            r'\d+\s+[A-Za-z0-9\s,]+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Lane|Ln|Drive|Dr)[,\s]+[A-Za-z\s]+,\s*[A-Z]{2}\s+\d{5}',
            r'\d+\s+[A-Za-z0-9\s,]+(?:Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Lane|Ln|Drive|Dr)[,\s]+[A-Za-z\s]+[,\s]+[A-Z]{2,}'
        ]

        text_content = soup.get_text()

        for pattern in address_patterns:
            found_addresses = re.findall(pattern, text_content)
            addresses.extend(found_addresses)

        # Look for address in specific elements
        address_elements = soup.find_all(['address', 'div'], class_=re.compile(r'address|location|contact', re.I))
        for element in address_elements:
            addresses.append(element.get_text().strip())

        return list(set(addresses))
    except Exception as e:
        logging.error(f"Error extracting addresses: {str(e)}")
        return []

def extract_market_position(self, soup: BeautifulSoup, html_content: str) -> Dict:
    """Extract market position information"""
    try:
        text_content = html_content.lower()

        position_indicators = {
            'leadership': [
                'market leader', 'industry leader', 'leading provider',
                'top provider', 'premier', 'best-in-class'
            ],
            'innovation': [
                'innovative', 'cutting-edge', 'pioneering',
                'revolutionary', 'next-generation'
            ],
            'recognition': [
                'award-winning', 'recognized by', 'acclaimed',
                'certified', 'trusted by'
            ]
        }

        market_position = {
            category: [
                indicator for indicator in indicators
                if indicator in text_content
            ]
            for category, indicators in position_indicators.items()
        }

        # Extract customer references
        customers = self.extract_customer_references(soup)
        if customers:
            market_position['notable_customers'] = customers

        return market_position
    except Exception as e:
        logging.error(f"Error extracting market position: {str(e)}")
        return {}

def extract_customer_references(self, soup: BeautifulSoup) -> List[str]:
    """Extract customer references"""
    try:
        customers = []

        # Look for customer logos
        logo_sections = soup.find_all(['div', 'section'], class_=re.compile(r'customer|client|logo', re.I))
        for section in logo_sections:
            images = section.find_all('img')
            for img in images:
                if img.get('alt'):
                    customers.append(img['alt'])

        # Look for customer mentions in text
        customer_sections = soup.find_all(['div', 'section'], class_=re.compile(r'customer|client|testimonial', re.I))
        for section in customer_sections:
            text = section.get_text()
            # Extract company names (simplified)
            company_matches = re.findall(r'(?:[A-Z][a-z]+\s+)+(?:Inc\.|LLC|Ltd\.?|Corporation|Corp\.?)', text)
            customers.extend(company_matches)

        return list(set(customers))
    except Exception as e:
        logging.error(f"Error extracting customer references: {str(e)}")
        return []

def extract_decision_makers(self, soup: BeautifulSoup, html_content: str) -> List[Dict]:
    """Extract information about decision makers"""
    try:
        decision_makers = []

        # Look for leadership/team sections
        team_sections = soup.find_all(['div', 'section'], class_=re.compile(r'team|leadership|management|executive', re.I))

        for section in team_sections:
            # Look for individual profiles
            profiles = section.find_all(['div', 'article'], class_=re.compile(r'profile|member|card', re.I))

            for profile in profiles:
                name = profile.find(['h2', 'h3', 'h4', 'strong'])
                title = profile.find(['p', 'div'], class_=re.compile(r'title|position|role', re.I))

                if name:
                    decision_maker = {
                        'name': name.get_text().strip(),
                        'title': title.get_text().strip() if title else '',
                        'department': self.determine_department(title.get_text().strip() if title else ''),
                        'seniority': self.determine_seniority(title.get_text().strip() if title else '')
                    }
                    decision_makers.append(decision_maker)

        return decision_makers
    except Exception as e:
        logging.error(f"Error extracting decision makers: {str(e)}")
        return []

def determine_department(self, title: str) -> str:
    """Determine department from title"""
    title_lower = title.lower()

    departments = {
        'technology': ['tech', 'engineering', 'development', 'it'],
        'sales': ['sales', 'revenue', 'business development'],
        'marketing': ['marketing', 'growth', 'brand'],
        'product': ['product', 'program'],
        'operations': ['operations', 'ops', 'operating'],
        'finance': ['finance', 'financial', 'accounting'],
        'hr': ['hr', 'human resources', 'people'],
        'executive': ['ceo', 'cto', 'cfo', 'coo', 'chief']
    }

    for dept, keywords in departments.items():
        if any(keyword in title_lower for keyword in keywords):
            return dept

    return 'other'

def determine_seniority(self, title: str) -> str:
    """Determine seniority level from title"""
    title_lower = title.lower()

    if any(c in title_lower for c in ['chief', 'ceo', 'cto', 'cfo', 'coo']):
        return 'c-level'
    elif any(word in title_lower for word in ['vp', 'vice president', 'svp', 'evp']):
        return 'vp-level'
    elif 'director' in title_lower:
        return 'director-level'
    elif 'manager' in title_lower:
        return 'manager-level'
    elif any(word in title_lower for word in ['lead', 'senior', 'sr']):
        return 'senior'
    else:
        return 'other'

In [ ]:
class EnhancedB2BProspector:
    """Enhanced B2B prospector for company analysis"""

    def __init__(self):
        self.data_extractor = EnhancedDataExtractor()
        self.decision_maker_identifier = DecisionMakerIdentifier()
        self.scoring_system = EnhancedScoring()
        self.session = None
        self.user_agent = UserAgent()
        self.request_timeout = 30
        self.max_retries = 3

    async def initialize(self):
        """Initialize components and session"""
        try:
            # Configure session with optimal settings
            self.session = aiohttp.ClientSession(
                timeout=aiohttp.ClientTimeout(total=self.request_timeout),
                headers={'User-Agent': self.user_agent.random}
            )
            logging.info("EnhancedB2BProspector initialized successfully")
        except Exception as e:
            logging.error(f"Error initializing B2B prospector: {str(e)}")
            raise

    async def cleanup(self):
        """Clean up resources"""
        if self.session and not self.session.closed:
            await self.session.close()
            logging.info("Resources cleaned up")

    async def analyze_company(self, domain: str) -> Dict:
        """Analyze a company by domain name"""
        try:
            logging.info(f"Starting analysis for {domain}")
            result = {
                'domain': domain,
                'success': False,
                'data': {},
                'errors': []
            }

            # Step 1: Fetch and parse website
            html_content, soup = await self.fetch_and_parse_website(domain)
            if not html_content:
                result['errors'].append("Failed to fetch website content")
                return result

            # Step 2: Extract base company profile
            company_profile = await self.extract_company_profile(domain, soup, html_content)

            # Step 3: Extract contact information
            contact_info = self.data_extractor.extract_contact_details(html_content)

            # Step 4: Identify decision makers
            decision_makers = await self.decision_maker_identifier.identify_decision_makers(html_content, soup)

            # Step 5: Extract technology stack
            tech_stack = await self.extract_technology_stack(domain, soup, html_content)

            # Step 6: Analyze business signals
            business_signals = await self.analyze_business_signals(domain, soup, html_content)

            # Combine all data
            combined_data = {
                'company_profile': company_profile,
                'contact_information': contact_info,
                'decision_makers': decision_makers,
                'technology_stack': tech_stack,
                'business_signals': business_signals,
                'soup': soup  # Include for further analysis if needed
            }

            # Step 7: Score the lead
            scoring_result = self.scoring_system.calculate_comprehensive_score(combined_data)
            combined_data['scoring'] = scoring_result

            result['data'] = combined_data
            result['success'] = True
            logging.info(f"Completed analysis for {domain}")

            return result
        except Exception as e:
            logging.error(f"Error analyzing {domain}: {str(e)}")
            result['errors'].append(str(e))
            return result

    async def fetch_and_parse_website(self, domain: str) -> Tuple[str, BeautifulSoup]:
        """Fetch website content and parse it"""
        url = f"https://{domain}"
        retry_count = 0

        while retry_count < self.max_retries:
            try:
                # Update headers with random user agent
                headers = {'User-Agent': self.user_agent.random}

                # Fetch main page
                async with self.session.get(url, headers=headers) as response:
                    if response.status == 200:
                        html_content = await response.text()
                        soup = BeautifulSoup(html_content, 'html.parser')
                        return html_content, soup
                    else:
                        logging.warning(f"Failed to fetch {url}, status: {response.status}")
                        retry_count += 1
                        await asyncio.sleep(1)
            except Exception as e:
                logging.error(f"Error fetching {url}: {str(e)}")
                retry_count += 1
                await asyncio.sleep(1)

        return "", None

    async def extract_company_profile(self, domain: str, soup: BeautifulSoup, html_content: str) -> Dict:
        """Extract company profile information"""
        try:
            company_profile = {
                'name': self.extract_company_name(soup),
                'domain': domain,
                'industry': self.extract_industry(soup, html_content),
                'company_size': self.extract_company_size(soup, html_content),
                'locations': self.extract_locations(soup),
                'description': self.extract_description(soup),
                'founding_year': self.extract_founding_year(soup, html_content)
            }
            return company_profile
        except Exception as e:
            logging.error(f"Error extracting company profile: {str(e)}")
            return {'domain': domain}

    def extract_company_name(self, soup: BeautifulSoup) -> str:
        """Extract company name from website"""
        try:
            # Try title tag first
            title = soup.title.string if soup.title else ""
            if title:
                # Clean up common title patterns
                company_name = title.split('|')[0].split('-')[0].strip()
                if len(company_name) > 3:
                    return company_name

            # Try meta tags
            meta_name = soup.find('meta', property='og:site_name')
            if meta_name and meta_name.get('content'):
                return meta_name['content'].strip()

            # Try logo alt text
            logo = soup.find('img', alt=True, src=re.compile(r'logo', re.I))
            if logo and logo.get('alt'):
                return logo['alt'].strip()

            # Try header text
            header = soup.find(['h1', 'h2'], class_=re.compile(r'logo|brand|company', re.I))
            if header:
                return header.get_text().strip()

            return "Unknown"
        except Exception as e:
            logging.error(f"Error extracting company name: {str(e)}")
            return "Unknown"

    def extract_industry(self, soup: BeautifulSoup, html_content: str) -> str:
        """Extract company industry from website"""
        try:
            # Common industry terms to look for
            industry_terms = [
                'technology', 'healthcare', 'finance', 'manufacturing',
                'retail', 'education', 'consulting', 'media'
            ]

            # Try meta description
            meta_desc = soup.find('meta', {'name': 'description'})
            if meta_desc and meta_desc.get('content'):
                desc = meta_desc['content'].lower()
                for industry in industry_terms:
                    if industry in desc:
                        return industry

            # Try about/company pages content
            for industry in industry_terms:
                if industry in html_content.lower():
                    return industry

            return "Unknown"
        except Exception as e:
            logging.error(f"Error extracting industry: {str(e)}")
            return "Unknown"

    def extract_company_size(self, soup: BeautifulSoup, html_content: str) -> str:
        """Extract company size from website"""
        try:
            # Look for employee count patterns
            employee_patterns = [
                r'([0-9,]+)\s+employees',
                r'team of\s+([0-9,]+)',
                r'over\s+([0-9,]+)\s+employees'
            ]

            for pattern in employee_patterns:
                matches = re.search(pattern, html_content, re.I)
                if matches:
                    count = int(matches.group(1).replace(',', ''))

                    if count < 50:
                        return 'small'
                    elif count < 500:
                        return 'mid_market'
                    else:
                        return 'enterprise'

            return "Unknown"
        except Exception as e:
            logging.error(f"Error extracting company size: {str(e)}")
            return "Unknown"

    def extract_locations(self, soup: BeautifulSoup) -> List[str]:
        """Extract company locations from website"""
        try:
            locations = []

            # Try address elements
            address_elems = soup.find_all(['address', 'div'], class_=re.compile(r'address|location', re.I))
            for addr in address_elems:
                text = addr.get_text().strip()
                if len(text) > 10:  # Basic validation
                    locations.append(text)

            # Try footer sections
            footer = soup.find('footer')
            if footer:
                address_pattern = r'\b[0-9]+\s+[A-Za-z\s]+,\s+[A-Za-z\s]+,\s+[A-Z]{2}\s+[0-9]{5}\b'
                addresses = re.findall(address_pattern, footer.get_text())
                locations.extend(addresses)

            return list(set(locations))
        except Exception as e:
            logging.error(f"Error extracting locations: {str(e)}")
            return []

    def extract_description(self, soup: BeautifulSoup) -> str:
        """Extract company description from website"""
        try:
            # Try meta description
            meta_desc = soup.find('meta', {'name': 'description'})
            if meta_desc and meta_desc.get('content'):
                return meta_desc['content'].strip()

            # Try about section
            about_section = soup.find(['div', 'section'], id=re.compile(r'about', re.I))
            if about_section:
                paragraphs = about_section.find_all('p')
                if paragraphs:
                    return paragraphs[0].get_text().strip()

            return ""
        except Exception as e:
            logging.error(f"Error extracting description: {str(e)}")
            return ""

    def extract_founding_year(self, soup: BeautifulSoup, html_content: str) -> str:
        """Extract founding year from website"""
        try:
            # Look for founding year patterns
            year_patterns = [
                r'[Ff]ounded in (\d{4})',
                r'[Ee]stablished in (\d{4})',
                r'[Ss]ince (\d{4})'
            ]

            for pattern in year_patterns:
                matches = re.search(pattern, html_content)
                if matches:
                    return matches.group(1)

            return ""
        except Exception as e:
            logging.error(f"Error extracting founding year: {str(e)}")
            return ""

    async def extract_technology_stack(self, domain: str, soup: BeautifulSoup, html_content: str) -> Dict:
        """Extract technology stack information"""
        try:
            # Initialize technology categories
            tech_stack = {
                'frontend_technologies': [],
                'backend_technologies': [],
                'cloud_services': [],
                'analytics_tools': [],
                'security_solutions': [],
                'cms_platforms': []
            }

            # Frontend technologies
            frontend_patterns = [
                r'React', r'Angular', r'Vue\.js', r'jQuery', r'Bootstrap',
                r'Tailwind', r'Gatsby', r'Next\.js', r'TypeScript'
            ]

            # Backend technologies
            backend_patterns = [
                r'Node\.js', r'Python', r'Ruby on Rails', r'Django', r'PHP',
                r'Java', r'Spring', r'ASP\.NET', r'Laravel', r'Express'
            ]

            # Cloud services
            cloud_patterns = [
                r'AWS', r'Amazon Web Services', r'Google Cloud', r'Azure',
                r'Cloudflare', r'DigitalOcean', r'Heroku'
            ]

            # Extract technologies from HTML source and scripts
            for pattern in frontend_patterns:
                if re.search(pattern, html_content, re.I):
                    tech_stack['frontend_technologies'].append(pattern)

            for pattern in backend_patterns:
                if re.search(pattern, html_content, re.I):
                    tech_stack['backend_technologies'].append(pattern)

            for pattern in cloud_patterns:
                if re.search(pattern, html_content, re.I):
                    tech_stack['cloud_services'].append(pattern)

            # Extract from script and link tags
            scripts = soup.find_all('script', src=True)
            for script in scripts:
                src = script.get('src', '')
                # Check common libraries and frameworks
                if 'react' in src.lower():
                    tech_stack['frontend_technologies'].append('React')
                elif 'angular' in src.lower():
                    tech_stack['frontend_technologies'].append('Angular')
                elif 'vue' in src.lower():
                    tech_stack['frontend_technologies'].append('Vue.js')

            # Identify CMS platforms
            cms_patterns = {
                'WordPress': [r'wp-content', r'wp-includes'],
                'Drupal': [r'drupal', r'sites/all'],
                'Joomla': [r'joomla'],
                'Shopify': [r'shopify']
            }

            for cms, patterns in cms_patterns.items():
                for pattern in patterns:
                    if re.search(pattern, html_content, re.I):
                        tech_stack['cms_platforms'].append(cms)
                        break

            return tech_stack
        except Exception as e:
            logging.error(f"Error extracting technology stack: {str(e)}")
            return {}

    async def analyze_business_signals(self, domain: str, soup: BeautifulSoup, html_content: str) -> Dict:
        """Analyze business signals from website"""
        try:
            signals = {
                'growth': [],
                'funding': [],
                'partnerships': [],
                'market_leadership': [],
                'technology_investment': []
            }

            # Growth signals
            growth_patterns = [
                r'expanding', r'growth', r'hiring', r'new office',
                r'increased revenue', r'growing team'
            ]

            # Funding signals
            funding_patterns = [
                r'funded', r'investment', r'series [ABC]', r'venture capital',
                r'raised', r'financing round'
            ]

            # Extract business signals from content
            for pattern in growth_patterns:
                if re.search(pattern, html_content, re.I):
                    signals['growth'].append(pattern)

            for pattern in funding_patterns:
                if re.search(pattern, html_content, re.I):
                    signals['funding'].append(pattern)

            # Check for partnership mentions
            partnership_section = soup.find(['div', 'section'],
                class_=re.compile(r'partners|customers|clients', re.I))
            if partnership_section:
                signals['partnerships'] = ['Has partnership section']

            # Check for technology investment signals
            tech_keywords = ['innovation', 'technology', 'digital transformation', 'automation']
            for keyword in tech_keywords:
                if keyword in html_content.lower():
                    signals['technology_investment'].append(keyword)

            return signals
        except Exception as e:
            logging.error(f"Error analyzing business signals: {str(e)}")
            return {}

In [ ]:
import asyncio
import aiohttp
from bs4 import BeautifulSoup
from typing import Dict, List, Tuple, Optional, Set
import pandas as pd
import logging
from fake_useragent import UserAgent
import time
import re
import json
from datetime import datetime

class EnhancedDataExtractor:
    """Enhanced data extraction patterns"""

    def __init__(self):
        self.email_blacklist = [
            'example.com', 'test.com', 'domain.com',
            'email.com', 'website.com', 'site.com'
        ]

        self.phone_formats = {
            'us': r'^\+?1?\s*$$?([0-9]{3})$$?[-.\s]?([0-9]{3})[-.\s]?([0-9]{4})$',
            'international': r'^\+?([0-9]{1,3})\s*$$?([0-9]{1,4})$$?[-.\s]?([0-9]{4,})$'
        }

    def extract_contact_details(self, html_content: str) -> Dict:
        """Enhanced contact detail extraction"""
        return {
            'emails': self.extract_emails_enhanced(html_content),
            'phones': self.extract_phones_enhanced(html_content),
            'social_profiles': self.extract_social_profiles_enhanced(html_content),
            'instant_messaging': self.extract_messaging_handles(html_content)
        }

    def extract_emails_enhanced(self, content: str) -> List[str]:
        """Enhanced email extraction with validation"""
        email_patterns = [
            r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',  # Standard email
            r'mailto:[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',  # Mailto links
            r'(?:contact|email|info|sales|support)@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'  # Common business emails
        ]

        emails = set()
        for pattern in email_patterns:
            found_emails = re.findall(pattern, content)
            emails.update(found_emails)

        return list(self.validate_emails(emails))

    def validate_emails(self, emails: Set[str]) -> Set[str]:
        """Validate extracted email addresses"""
        valid_emails = set()
        email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'

        for email in emails:
            # Remove 'mailto:' if present
            email = email.replace('mailto:', '').strip()

            # Basic pattern validation
            if not re.match(email_pattern, email):
                continue

            # Check domain blacklist
            domain = email.split('@')[1].lower()
            if domain in self.email_blacklist:
                continue

            # Additional validation rules
            if len(email) > 7 and '.' in domain:  # Basic sanity check
                valid_emails.add(email.lower())

        return valid_emails

    def extract_phones_enhanced(self, content: str) -> List[str]:
        """Enhanced phone number extraction"""
        phone_patterns = [
            r'\+\d{1,3}[-.\s]?\d{1,3}[-.\s]?\d{3,4}[-.\s]?\d{3,4}',  # International
            r'$$\d{3}$$[-.\s]?\d{3}[-.\s]?\d{4}',  # US/Canada
            r'\d{3}[-.\s]?\d{3}[-.\s]?\d{4}',  # Simple
            r'(?:tel|phone|call)[:\s]+[+\d\s()-]{10,}'  # With prefixes
        ]

        phones = set()
        for pattern in phone_patterns:
            found_phones = re.findall(pattern, content)
            phones.update(found_phones)

        return list(self.clean_phone_numbers(phones))

    def clean_phone_numbers(self, phones: Set[str]) -> Set[str]:
        """Clean and validate phone numbers"""
        clean_phones = set()

        for phone in phones:
            # Remove common prefixes
            phone = re.sub(r'(?:tel|phone|call)[:\s]+', '', phone, flags=re.I)

            # Remove non-digit characters except + for international
            cleaned = re.sub(r'[^\d+]', '', phone)

            # Validate length
            if len(cleaned) >= 10:  # Most phone numbers are at least 10 digits
                clean_phones.add(cleaned)

        return clean_phones

    def extract_social_profiles_enhanced(self, html_content: str) -> Dict[str, str]:
        """Extract social media profiles"""
        social_patterns = {
            'linkedin': r'linkedin\.com/(?:company/[^"\s]+|in/[^"\s]+)',
            'twitter': r'twitter\.com/[^"\s]+',
            'facebook': r'facebook\.com/[^"\s]+',
            'instagram': r'instagram\.com/[^"\s]+',
            'youtube': r'youtube\.com/(?:channel/|user/)?[^"\s]+',
            'github': r'github\.com/[^"\s]+'
        }

        profiles = {}
        for platform, pattern in social_patterns.items():
            matches = re.findall(pattern, html_content, re.I)
            if matches:
                profiles[platform] = self.clean_social_url(matches[0])

        return profiles

    def clean_social_url(self, url: str) -> str:
        """Clean social media URLs"""
        # Remove any URL parameters
        url = url.split('?')[0]
        # Ensure https://
        if not url.startswith('http'):
            url = 'https://' + url
        return url

    def extract_messaging_handles(self, html_content: str) -> Dict[str, str]:
        """Extract instant messaging handles"""
        messaging_patterns = {
            'skype': r'skype:([^"\s]+)',
            'telegram': r't\.me/([^"\s]+)',
            'whatsapp': r'wa\.me/([^"\s]+)',
            'slack': r'slack\.com/([^"\s]+)'
        }

        handles = {}
        for platform, pattern in messaging_patterns.items():
            matches = re.findall(pattern, html_content, re.I)
            if matches:
                handles[platform] = matches[0]

        return handles

In [ ]:
class DecisionMakerIdentifier:
    """Enhanced decision maker identification"""

    def __init__(self):
        self.title_patterns = {
            'executive_level': [
                r'C[A-Z][O]',  # CXO patterns
                r'Chief\s+[A-Z]\w+\s+Officer',
                r'President',
                r'Founder',
                r'Owner'
            ],
            'senior_management': [
                r'Senior\s+Vice\s+President',
                r'VP\s+of\s+[A-Z]\w+',
                r'Head\s+of\s+[A-Z]\w+',
                r'Director\s+of\s+[A-Z]\w+'
            ],
            'technical_leaders': [
                r'Chief\s+Technology\s+Officer',
                r'Chief\s+Information\s+Officer',
                r'Technical\s+Director',
                r'Engineering\s+Manager'
            ]
        }

    async def identify_decision_makers(self, html_content: str, soup: BeautifulSoup) -> List[Dict]:
        """Identify decision makers from content"""
        decision_makers = []

        # Extract from team/about pages
        team_sections = soup.find_all(['div', 'section'],
            class_=re.compile(r'team|leadership|management|about', re.I))

        for section in team_sections:
            decision_makers.extend(self.extract_from_section(section))

        # Extract from LinkedIn data if available
        linkedin_data = await self.extract_linkedin_data(html_content)
        if linkedin_data:
            decision_makers.extend(linkedin_data)

        return self.deduplicate_decision_makers(decision_makers)

    def extract_from_section(self, section: BeautifulSoup) -> List[Dict]:
        """Extract decision makers from a section"""
        decision_makers = []

        # Look for common team member patterns
        team_members = section.find_all(['div', 'article'],
            class_=re.compile(r'member|profile|card|person', re.I))

        for member in team_members:
            dm_info = self.extract_member_info(member)
            if dm_info:
                decision_makers.append(dm_info)

        return decision_makers

    def extract_member_info(self, member: BeautifulSoup) -> Optional[Dict]:
        """Extract information about a team member"""
        try:
            name_elem = member.find(['h2', 'h3', 'h4', 'strong'],
                class_=re.compile(r'name|title', re.I))
            title_elem = member.find(['p', 'div', 'span'],
                class_=re.compile(r'title|position|role', re.I))

            if name_elem:
                name = name_elem.get_text().strip()
                title = title_elem.get_text().strip() if title_elem else ''

                return {
                    'name': name,
                    'title': title,
                    'level': self.determine_seniority_level(title),
                    'department': self.determine_department(title),
                    'decision_making_power': self.calculate_decision_power(title)
                }
        except Exception as e:
            logging.error(f"Error extracting member info: {str(e)}")
        return None

    async def extract_linkedin_data(self, html_content: str) -> List[Dict]:
        """Extract decision maker info from LinkedIn data"""
        try:
            linkedin_patterns = {
                'profile': r'linkedin\.com/in/([^"\s]+)',
                'title': r'title=["\'](.*?)["\']'
            }

            profiles = re.findall(linkedin_patterns['profile'], html_content)
            titles = re.findall(linkedin_patterns['title'], html_content)

            linkedin_data = []
            for profile, title in zip(profiles, titles):
                if self.is_decision_maker(title):
                    linkedin_data.append({
                        'source': 'linkedin',
                        'profile': f"linkedin.com/in/{profile}",
                        'title': title,
                        'level': self.determine_seniority_level(title),
                        'department': self.determine_department(title)
                    })

            return linkedin_data
        except Exception as e:
            logging.error(f"Error extracting LinkedIn data: {str(e)}")
            return []

    def determine_seniority_level(self, title: str) -> str:
        """Determine seniority level from title"""
        title_lower = title.lower()

        if any(pattern.lower() in title_lower for pattern in self.title_patterns['executive_level']):
            return 'executive'
        elif any(pattern.lower() in title_lower for pattern in self.title_patterns['senior_management']):
            return 'senior'
        elif any(pattern.lower() in title_lower for pattern in self.title_patterns['technical_leaders']):
            return 'technical_leader'
        else:
            return 'other'

    def determine_department(self, title: str) -> str:
        """Determine department from title"""
        departments = {
            'technology': ['engineering', 'technical', 'technology', 'IT', 'development'],
            'sales': ['sales', 'revenue', 'business development'],
            'marketing': ['marketing', 'growth', 'brand'],
            'product': ['product', 'program'],
            'operations': ['operations', 'ops'],
            'finance': ['finance', 'financial'],
            'hr': ['HR', 'human resources', 'people']
        }

        title_lower = title.lower()
        for dept, keywords in departments.items():
            if any(keyword.lower() in title_lower for keyword in keywords):
                return dept
        return 'other'

    def calculate_decision_power(self, title: str) -> float:
        """Calculate decision-making power score"""
        score = 0.0
        title_lower = title.lower()

        # Position-based scoring
        if any(c in title_lower for c in ['ceo', 'cto', 'cio', 'cfo']):
            score += 1.0
        elif any(word in title_lower for word in ['vp', 'vice president', 'director']):
            score += 0.8
        elif any(word in title_lower for word in ['head', 'lead', 'senior']):
            score += 0.6
        elif 'manager' in title_lower:
            score += 0.4

        # Department-based adjustment
        department = self.determine_department(title)
        if department in ['technology', 'sales', 'product']:
            score *= 1.2

        return min(score, 1.0)

    def deduplicate_decision_makers(self, decision_makers: List[Dict]) -> List[Dict]:
        """Remove duplicate decision makers"""
        unique_dms = {}
        for dm in decision_makers:
            key = f"{dm.get('name', '')}-{dm.get('title', '')}"
            if key not in unique_dms or dm.get('decision_making_power', 0) > unique_dms[key].get('decision_making_power', 0):
                unique_dms[key] = dm
        return list(unique_dms.values())

    def is_decision_maker(self, title: str) -> bool:
        """Determine if a title represents a decision maker"""
        title_lower = title.lower()

        decision_maker_indicators = [
            'c-level',
            'chief',
            'vp',
            'vice president',
            'director',
            'head of',
            'lead',
            'manager'
        ]

        return any(indicator in title_lower for indicator in decision_maker_indicators)

In [ ]:
class EnhancedScoring:
    """Enhanced lead scoring system"""

    def __init__(self):
        self.scoring_weights = {
            'company_fit': 0.25,
            'technical_fit': 0.20,
            'decision_maker_presence': 0.20,
            'engagement_potential': 0.15,
            'budget_indicators': 0.10,
            'timing_indicators': 0.10
        }

        self.industry_priorities = {
            'technology': 1.2,
            'finance': 1.1,
            'healthcare': 1.1,
            'manufacturing': 1.0,
            'retail': 0.9
        }

    def calculate_comprehensive_score(self, data: Dict) -> Dict:
        """Calculate comprehensive lead score"""
        try:
            scores = {
                'company_fit': self.calculate_company_fit(data),
                'technical_fit': self.calculate_technical_fit(data),
                'decision_maker_presence': self.calculate_decision_maker_score(data),
                'engagement_potential': self.calculate_engagement_potential(data),
                'budget_indicators': self.calculate_budget_indicators(data),
                'timing_indicators': self.calculate_timing_indicators(data)
            }

            weighted_score = self.calculate_weighted_score(scores)

            return {
                'total_score': weighted_score,
                'component_scores': scores,
                'analysis': self.analyze_scores(scores),
                'recommendations': self.generate_recommendations(scores, data)
            }
        except Exception as e:
            logging.error(f"Error calculating comprehensive score: {str(e)}")
            return self.get_default_score()

    def calculate_company_fit(self, data: Dict) -> float:
        """Calculate company fit score"""
        score = 0.0
        company_profile = data.get('company_profile', {})

        # Industry fit
        industry = company_profile.get('industry', '').lower()
        industry_multiplier = self.industry_priorities.get(industry, 1.0)
        score += 0.3 * industry_multiplier

        # Size fit
        size = company_profile.get('company_size', '').lower()
        if size in ['enterprise', 'mid_market']:
            score += 0.3
        elif size == 'smb':
            score += 0.2

        # Market position
        market_position = data.get('market_position', {})
        if market_position.get('market_leadership', []):
            score += 0.2

        # Geographic presence
        if company_profile.get('locations', []):
            score += 0.2

        return min(score, 1.0)

    def calculate_technical_fit(self, data: Dict) -> float:
        """Calculate technical fit score"""
        score = 0.0
        tech_stack = data.get('technology_stack', {})

        # Modern technology adoption
        modern_tech_score = len(tech_stack.get('frontend_technologies', [])) * 0.1 + \
                          len(tech_stack.get('backend_technologies', [])) * 0.1
        score += min(modern_tech_score, 0.4)

        # Cloud services
        if tech_stack.get('cloud_services', []):
            score += 0.3

        # Security solutions
        if tech_stack.get('security_solutions', []):
            score += 0.3

        return min(score, 1.0)

    def calculate_decision_maker_score(self, data: Dict) -> float:
        """Calculate decision maker presence score"""
        score = 0.0
        decision_makers = data.get('decision_makers', [])

        # Score based on seniority levels present
        seniority_scores = {
            'executive': 0.4,
            'senior': 0.3,
            'technical_leader': 0.2,
            'other': 0.1
        }

        for dm in decision_makers:
            level = dm.get('level', 'other')
            score += seniority_scores.get(level, 0)

        return min(score, 1.0)

    def calculate_engagement_potential(self, data: Dict) -> float:
        """Calculate engagement potential score"""
        score = 0.0
        contact_info = data.get('contact_information', {})

        # Contact information completeness
        if contact_info.get('emails'):
            score += 0.3
        if contact_info.get('phones'):
            score += 0.2
        if contact_info.get('social_profiles'):
            score += 0.2

        # Recent engagement indicators
        if data.get('recent_interactions', []):
            score += 0.3

        return min(score, 1.0)

    def calculate_budget_indicators(self, data: Dict) -> float:
        """Calculate budget indicators score"""
        score = 0.0
        company_profile = data.get('company_profile', {})

        # Company size as budget indicator
        size = company_profile.get('company_size', '').lower()
        if size == 'enterprise':
            score += 0.4
        elif size == 'mid_market':
            score += 0.3
        elif size == 'smb':
            score += 0.2

        # Technology investment signals
        tech_investment = data.get('business_signals', {}).get('technology_investment', [])
        if tech_investment:
            score += 0.3

        # Growth signals as budget indicators
        growth_signals = data.get('business_signals', {}).get('growth', [])
        if growth_signals:
            score += 0.3

        return min(score, 1.0)

    def calculate_timing_indicators(self, data: Dict) -> float:
        """Calculate timing indicators score"""
        score = 0.0
        business_signals = data.get('business_signals', {})

        # Growth and expansion signals
        if business_signals.get('growth'):
            score += 0.3

        # Technology investment signals
        if business_signals.get('technology_investment'):
            score += 0.3

        # Recent activities
        if data.get('recent_activities', []):
            score += 0.4

        return min(score, 1.0)

    def calculate_weighted_score(self, scores: Dict) -> float:
        """Calculate weighted total score"""
        weighted_score = sum(
            score * self.scoring_weights.get(component, 0)
            for component, score in scores.items()
        )
        return min(weighted_score, 1.0)

    def analyze_scores(self, scores: Dict) -> List[str]:
        """Generate analysis based on scores"""
        analysis = []

        for component, score in scores.items():
            if score >= 0.8:
                analysis.append(f"Strong {component.replace('_', ' ')} indicates high potential")
            elif score <= 0.3:
                analysis.append(f"Low {component.replace('_', ' ')} might need attention")

        return analysis

    def generate_recommendations(self, scores: Dict, data: Dict) -> List[str]:
        """Generate recommendations based on scores and data"""
        recommendations = []

        # High-priority recommendations
        if scores['company_fit'] > 0.7 and scores['decision_maker_presence'] > 0.6:
            recommendations.append("High-priority target - Immediate engagement recommended")

        # Technical validation needed
        if scores['technical_fit'] < 0.4:
            recommendations.append("Technical validation needed - Involve solution architect")

        # Missing decision makers
        if scores['decision_maker_presence'] < 0.3:
            recommendations.append("Identify and engage with more decision makers")

        # Engagement strategy
        if scores['engagement_potential'] > 0.7:
            recommendations.append("Ready for direct engagement - Multiple contact points available")

        return recommendations

    def get_default_score(self) -> Dict:
        """Return default score structure"""
        return {
            'total_score': 0.0,
            'component_scores': {
                component: 0.0 for component in self.scoring_weights.keys()
            },
            'analysis': [],
            'recommendations': []
        }

In [ ]:
class DetailedAnalyzer:
    """Enhanced analysis capabilities"""

    def generate_detailed_analysis(self, data: Dict) -> Dict:
        """Generate detailed analysis of all data points"""
        try:
            return {
                'company_analysis': self.analyze_company_data(data),
                'technical_analysis': self.analyze_technical_stack(data),
                'contact_analysis': self.analyze_contact_data(data),
                'engagement_analysis': self.analyze_engagement_potential(data),
                'risk_analysis': self.analyze_risks(data),
                'opportunity_analysis': self.analyze_opportunities(data)
            }
        except Exception as e:
            logging.error(f"Error generating detailed analysis: {str(e)}")
            return {}

    def analyze_company_data(self, data: Dict) -> Dict:
        """Analyze company data"""
        company_profile = data.get('company_profile', {})
        return {
            'size_category': self.determine_size_category(company_profile),
            'industry_focus': self.analyze_industry_focus(company_profile),
            'market_presence': self.analyze_market_presence(data),
            'growth_indicators': self.analyze_growth_indicators(data)
        }

    def analyze_technical_stack(self, data: Dict) -> Dict:
        """Analyze technical stack"""
        tech_stack = data.get('technology_stack', {})
        return {
            'stack_maturity': self.assess_stack_maturity(tech_stack),
            'modern_tech_adoption': self.assess_tech_adoption(tech_stack),
            'integration_potential': self.assess_integration_potential(tech_stack),
            'technical_gaps': self.identify_technical_gaps(tech_stack)
        }

    def analyze_contact_data(self, data: Dict) -> Dict:
        """Analyze contact information"""
        contact_info = data.get('contact_information', {})
        return {
            'contact_completeness': self.assess_contact_completeness(contact_info),
            'decision_maker_coverage': self.assess_decision_maker_coverage(data),
            'engagement_channels': self.identify_engagement_channels(contact_info),
            'contact_quality': self.assess_contact_quality(contact_info)
        }

    def analyze_engagement_potential(self, data: Dict) -> Dict:
        """Analyze engagement potential"""
        return {
            'readiness_score': self.calculate_readiness_score(data),
            'best_channels': self.identify_best_channels(data),
            'engagement_barriers': self.identify_barriers(data),
            'next_steps': self.recommend_next_steps(data)
        }

    def analyze_risks(self, data: Dict) -> Dict:
        """Analyze potential risks"""
        return {
            'competitive_risks': self.identify_competitive_risks(data),
            'timing_risks': self.identify_timing_risks(data),
            'budget_risks': self.identify_budget_risks(data),
            'technical_risks': self.identify_technical_risks(data)
        }

    def analyze_opportunities(self, data: Dict) -> Dict:
        """Analyze opportunities"""
        return {
            'immediate_opportunities': self.identify_immediate_opportunities(data),
            'growth_potential': self.assess_growth_potential(data),
            'upsell_opportunities': self.identify_upsell_opportunities(data),
            'partnership_potential': self.assess_partnership_potential(data)
        }

class DataPresenter:
    """Enhanced data presentation"""

    def generate_detailed_report(self, data: Dict) -> str:
        """Generate detailed report of all extracted information"""
        try:
            report_sections = [
                self.format_company_info(data),
                self.format_contact_info(data),
                self.format_technical_info(data),
                self.format_decision_makers(data),
                self.format_analysis(data)
            ]

            return '\n\n'.join(filter(None, report_sections))
        except Exception as e:
            logging.error(f"Error generating report: {str(e)}")
            return "Error generating report"

    def format_company_info(self, data: Dict) -> str:
        """Format company information"""
        company_info = data.get('company_profile', {})
        return f"""
Company Information:
------------------
Name: {company_info.get('name', 'N/A')}
Industry: {company_info.get('industry', 'N/A')}
Size: {company_info.get('size', 'N/A')}
Location(s): {', '.join(company_info.get('locations', ['N/A']))}
Market Position: {company_info.get('market_position', 'N/A')}
"""

    def format_contact_info(self, data: Dict) -> str:
        """Format contact information"""
        contact_info = data.get('contact_information', {})
        return f"""
Contact Information:
------------------
Emails: {', '.join(contact_info.get('emails', ['None found']))}
Phone Numbers: {', '.join(contact_info.get('phones', ['None found']))}
Addresses: {', '.join(contact_info.get('addresses', ['None found']))}
Social Profiles: {self.format_social_profiles(contact_info.get('social_profiles', {}))}
"""

    def format_technical_info(self, data: Dict) -> str:
        """Format technical information"""
        tech_stack = data.get('technology_stack', {})
        if not tech_stack:
            return ""

        tech_sections = []
        for category, technologies in tech_stack.items():
            if technologies:
                tech_sections.append(f"{category.replace('_', ' ').title()}:")
                tech_sections.extend([f"  • {tech}" for tech in technologies])

        return "\nTechnology Stack:\n------------------\n" + "\n".join(tech_sections)

    def format_decision_makers(self, data: Dict) -> str:
        """Format decision maker information"""
        decision_makers = data.get('decision_makers', [])
        if not decision_makers:
            return ""

        dm_sections = ["Decision Makers:\n------------------"]
        for dm in decision_makers:
            dm_sections.append(
                f"• {dm.get('name', 'N/A')} - {dm.get('title', 'N/A')}\n"
                f"  Level: {dm.get('level', 'N/A')}\n"
                f"  Department: {dm.get('department', 'N/A')}"
            )

        return "\n".join(dm_sections)

    def format_analysis(self, data: Dict) -> str:
        """Format analysis information"""
        analysis = data.get('analysis', {})
        if not analysis:
            return ""

        analysis_sections = ["Analysis:\n------------------"]
        for category, details in analysis.items():
            analysis_sections.append(f"\n{category.replace('_', ' ').title()}:")
            if isinstance(details, dict):
                for key, value in details.items():
                    analysis_sections.append(f"  • {key}: {value}")
            elif isinstance(details, list):
                analysis_sections.extend([f"  • {item}" for item in details])
            else:
                analysis_sections.append(f"  • {details}")

        return "\n".join(analysis_sections)

    def format_social_profiles(self, profiles: Dict) -> str:
        """Format social profile information"""
        if not profiles:
            return "None found"

        return "\n  ".join([f"{platform}: {url}" for platform, url in profiles.items()])

# Main execution code
async def main():
    # Initialize components
    prospector = EnhancedB2BProspector()
    await prospector.initialize()

    # Test domains
    domains = [
        'microsoft.com',
        'salesforce.com',
        'apple.com'
    ]

    results = []
    try:
        for domain in domains:
            print(f"\nAnalyzing {domain}...")
            result = await prospector.analyze_company(domain)
            results.append(result)
            print(f"Completed analysis of {domain}")
            await asyncio.sleep(2)  # Rate limiting

        return results
    finally:
        await prospector.cleanup()

async def run_analysis():
    try:
        results = await main()

        print("\nGenerating Detailed Analysis...")
        analyzer = DetailedAnalyzer()
        presenter = DataPresenter()

        for result in results:
            if result['success']:
                print(f"\nDetailed Analysis for {result['domain']}")
                print("="*50)

                # Generate and display detailed report
                detailed_report = presenter.generate_detailed_report(result['data'])
                print(detailed_report)

                # Generate and display enhanced analysis
                analysis = analyzer.generate_detailed_analysis(result['data'])
                print("\nEnhanced Analysis:")
                print("-"*30)
                for category, details in analysis.items():
                    print(f"\n{category.replace('_', ' ').title()}:")
                    for key, value in details.items():
                        print(f"  {key}: {value}")

                print("="*50)

        # Save results
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        with open(f'detailed_analysis_{timestamp}.json', 'w') as f:
            # Convert soup objects to string for JSON serialization
            serializable_results = []
            for result in results:
                if result['success']:
                    result['data']['soup'] = str(result['data']['soup'])
                serializable_results.append(result)
            json.dump(serializable_results, f, indent=2)

        print(f"\nDetailed results saved to detailed_analysis_{timestamp}.json")

        return results
    except Exception as e:
        print(f"Error in analysis: {str(e)}")
        return []

# Execute the analysis
if __name__ == "__main__":
    results = asyncio.run(run_analysis())


Analyzing microsoft.com...
Completed analysis of microsoft.com

Analyzing salesforce.com...


ERROR:root:Error extracting company size: invalid literal for int() with base 10: ''


Completed analysis of salesforce.com

Analyzing apple.com...
Completed analysis of apple.com


ERROR:root:Error generating detailed analysis: 'DetailedAnalyzer' object has no attribute 'determine_size_category'
ERROR:root:Error generating detailed analysis: 'DetailedAnalyzer' object has no attribute 'determine_size_category'
ERROR:root:Error generating detailed analysis: 'DetailedAnalyzer' object has no attribute 'determine_size_category'



Generating Detailed Analysis...

Detailed Analysis for microsoft.com

Company Information:
------------------
Name: Your request has been blocked. This could be
                        due to several reasons.
Industry: manufacturing
Size: N/A
Location(s): 
Market Position: N/A



Contact Information:
------------------
Emails: 
Phone Numbers: 
Addresses: None found
Social Profiles: None found



Technology Stack:
------------------
Frontend Technologies:
  • jQuery
Backend Technologies:
  • Java
Cloud Services:
  • Azure

Enhanced Analysis:
------------------------------

Detailed Analysis for salesforce.com

Company Information:
------------------
Name: Salesforce: The Customer Company
Industry: technology
Size: N/A
Location(s): 
Market Position: N/A



Contact Information:
------------------
Emails: 
Phone Numbers: 1724186268, 1721867649, 1727982625, 1727982660, 1722968223, 1731439402, 1731438979, 1722982905, 1722983105, 1723063747, 1736902407, 1721341393, 0000000000, 1723066247, 72

In [ ]:
import asyncio
import aiohttp
from bs4 import BeautifulSoup
from typing import Dict, List, Tuple, Optional, Set
import pandas as pd
import logging
from fake_useragent import UserAgent
import time
import re
from datetime import datetime

class EnhancedB2BProspector:
    def __init__(self):
        self.headers = {
            'User-Agent': UserAgent().random,
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        }
        self.session = None
        self.business_signals = {
            'company_size_indicators': {
                'enterprise': [
                    'global presence', 'enterprise solutions', 'fortune 500',
                    'multiple offices', 'worldwide locations', 'enterprise-grade'
                ],
                'mid_market': [
                    'growing company', 'medium-sized', 'regional leader',
                    'expanding business', 'multiple locations'
                ],
                'smb': [
                    'small business', 'startup', 'local business',
                    'family owned', 'newly established'
                ]
            },
            'business_indicators': {
                'growth': [
                    'expanding', 'growing', 'hiring', 'new office',
                    'recent funding', 'series [abcdef]', 'acquisition'
                ],
                'technology_investment': [
                    'digital transformation', 'modernization', 'automation',
                    'ai implementation', 'cloud migration', 'platform upgrade'
                ],
                'market_position': [
                    'market leader', 'industry pioneer', 'award-winning',
                    'recognized by', 'leading provider', 'trusted by'
                ]
            }
        }

    async def initialize(self):
        if not self.session:
            self.session = aiohttp.ClientSession(headers=self.headers)

    async def cleanup(self):
        if self.session:
            await self.session.close()

In [ ]:
    async def analyze_company(self, domain: str) -> Dict:
        try:
            url = f"https://www.{domain}"
            soup, html_content = await self.fetch_page(url)

            if not soup or not html_content:
                return self.get_empty_result(domain)

            data = await self.extract_b2b_data(soup, html_content)
            data['domain'] = domain
            data['url'] = url
            data['success'] = True

            return data

        except Exception as e:
            logging.error(f"Error analyzing {domain}: {str(e)}")
            return self.get_empty_result(domain)

    async def extract_b2b_data(self, soup: BeautifulSoup, html_content: str) -> Dict:
        try:
            return {
                'company_profile': await self.extract_company_profile(soup),
                'contact_information': self.extract_contact_information(soup, html_content),
                'technology_stack': self.extract_technology_stack(html_content),
                'business_signals': self.extract_business_signals(html_content),
                'market_position': self.extract_market_position(soup, html_content),
                'decision_makers': self.extract_decision_makers(soup, html_content)
            }
        except Exception as e:
            logging.error(f"Error extracting B2B data: {str(e)}")
            return self.get_empty_data_structure()

    async def extract_company_profile(self, soup: BeautifulSoup) -> Dict:
        try:
            return {
                'name': self.extract_company_name(soup),
                'industry': self.extract_industry(soup),
                'company_size': self.determine_company_size(soup),
                'locations': self.extract_locations(soup),
                'products_services': self.extract_products_services(soup),
                'description': self.extract_company_description(soup),
                'founded_year': self.extract_founded_year(soup),
                'employee_count': self.extract_employee_count(soup)
            }
        except Exception as e:
            logging.error(f"Error extracting company profile: {str(e)}")
            return self.get_empty_company_profile()

    def extract_contact_information(self, soup: BeautifulSoup, html_content: str) -> Dict:
        try:
            return {
                'emails': self.extract_emails(html_content),
                'phones': self.extract_phones(html_content),
                'addresses': self.extract_addresses(soup),
                'social_links': self.extract_social_links(soup)
            }
        except Exception as e:
            logging.error(f"Error extracting contact information: {str(e)}")
            return {'emails': [], 'phones': [], 'addresses': [], 'social_links': {}}

    def extract_technology_stack(self, html_content: str) -> Dict:
        try:
            return {
                'analytics': self.detect_analytics_tools(html_content),
                'crm': self.detect_crm_systems(html_content),
                'marketing': self.detect_marketing_tools(html_content),
                'development': self.detect_development_tech(html_content)
            }
        except Exception as e:
            logging.error(f"Error extracting technology stack: {str(e)}")
            return {}

    def extract_market_position(self, soup: BeautifulSoup, html_content: str) -> Dict:
        try:
            text_content = html_content.lower()
            position_data = {
                'market_leadership': [],
                'competitive_advantages': [],
                'customer_segments': [],
                'geographic_presence': []
            }

            # Market leadership indicators
            leadership_indicators = [
                'market leader', 'industry leader', 'leading provider',
                'top provider', 'premier', 'best-in-class'
            ]
            position_data['market_leadership'] = [
                indicator for indicator in leadership_indicators
                if indicator in text_content
            ]

            return position_data
        except Exception as e:
            logging.error(f"Error extracting market position: {str(e)}")
            return {}

    def extract_decision_makers(self, soup: BeautifulSoup, html_content: str) -> List[Dict]:
        try:
            decision_makers = []

            # Look for team/leadership sections
            team_sections = soup.find_all(['div', 'section'],
                class_=re.compile(r'team|leadership|management|about', re.I))

            for section in team_sections:
                decision_makers.extend(self.extract_team_members(section))

            return decision_makers
        except Exception as e:
            logging.error(f"Error extracting decision makers: {str(e)}")
            return []

In [ ]:
    def extract_company_name(self, soup: BeautifulSoup) -> str:
        try:
            # Try meta tags first
            meta_name = soup.find('meta', property='og:site_name')
            if meta_name and meta_name.get('content'):
                return meta_name['content'].strip()

            # Try title
            title = soup.find('title')
            if title:
                return title.text.split('|')[0].strip()

            return ''
        except Exception as e:
            logging.error(f"Error extracting company name: {str(e)}")
            return ''

    def extract_industry(self, soup: BeautifulSoup) -> str:
        try:
            text_content = soup.get_text().lower()
            industries = {
                'technology': ['software', 'technology', 'it services', 'cloud', 'digital'],
                'finance': ['financial', 'banking', 'insurance', 'fintech'],
                'healthcare': ['healthcare', 'medical', 'biotech', 'pharma'],
                'manufacturing': ['manufacturing', 'industrial', 'production'],
                'retail': ['retail', 'ecommerce', 'shopping', 'consumer']
            }

            for industry, keywords in industries.items():
                if any(keyword in text_content for keyword in keywords):
                    return industry
            return 'unknown'
        except Exception as e:
            logging.error(f"Error extracting industry: {str(e)}")
            return 'unknown'

    def determine_company_size(self, soup: BeautifulSoup) -> str:
        try:
            text_content = soup.get_text().lower()

            enterprise_indicators = ['enterprise', 'global', 'fortune 500', 'worldwide']
            midmarket_indicators = ['mid-market', 'growing company', 'regional']
            smb_indicators = ['small business', 'startup', 'local business']

            if any(indicator in text_content for indicator in enterprise_indicators):
                return 'enterprise'
            elif any(indicator in text_content for indicator in midmarket_indicators):
                return 'mid_market'
            elif any(indicator in text_content for indicator in smb_indicators):
                return 'smb'
            return 'unknown'
        except Exception as e:
            logging.error(f"Error determining company size: {str(e)}")
            return 'unknown'

    def extract_locations(self, soup: BeautifulSoup) -> List[str]:
        try:
            locations = []
            address_elements = soup.find_all(['address', 'div'],
                class_=re.compile('address|location', re.I))

            for element in address_elements:
                locations.append(element.get_text().strip())

            return list(set(locations))
        except Exception as e:
            logging.error(f"Error extracting locations: {str(e)}")
            return []

    def extract_emails(self, html_content: str) -> List[str]:
        try:
            email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
            emails = re.findall(email_pattern, html_content)
            return list(set(emails))
        except Exception as e:
            logging.error(f"Error extracting emails: {str(e)}")
            return []

    def extract_phones(self, html_content: str) -> List[str]:
        try:
            phone_patterns = [
                r'\+\d{1,3}[-.\s]?\d{1,3}[-.\s]?\d{3,4}[-.\s]?\d{3,4}',
                r'$$\d{3}$$[-.\s]?\d{3}[-.\s]?\d{4}',
                r'\d{3}[-.\s]?\d{3}[-.\s]?\d{4}'
            ]
            phones = []
            for pattern in phone_patterns:
                phones.extend(re.findall(pattern, html_content))
            return list(set(phones))
        except Exception as e:
            logging.error(f"Error extracting phones: {str(e)}")
            return []

    def extract_social_links(self, soup: BeautifulSoup) -> Dict[str, str]:
        try:
            social_patterns = {
                'linkedin': r'linkedin\.com/(?:company|in)/',
                'twitter': r'twitter\.com/',
                'facebook': r'facebook\.com/',
                'instagram': r'instagram\.com/'
            }

            social_links = {}
            for platform, pattern in social_patterns.items():
                links = soup.find_all('a', href=re.compile(pattern))
                if links:
                    social_links[platform] = links[0]['href']
            return social_links
        except Exception as e:
            logging.error(f"Error extracting social links: {str(e)}")
            return {}

    def detect_analytics_tools(self, html_content: str) -> List[str]:
        try:
            tools = ['google analytics', 'mixpanel', 'segment', 'amplitude']
            return [tool for tool in tools if tool in html_content.lower()]
        except Exception as e:
            logging.error(f"Error detecting analytics tools: {str(e)}")
            return []

    def detect_crm_systems(self, html_content: str) -> List[str]:
        try:
            systems = ['salesforce', 'hubspot', 'zoho', 'pipedrive']
            return [system for system in systems if system in html_content.lower()]
        except Exception as e:
            logging.error(f"Error detecting CRM systems: {str(e)}")
            return []

    def detect_marketing_tools(self, html_content: str) -> List[str]:
        try:
            tools = ['marketo', 'mailchimp', 'sendgrid', 'intercom']
            return [tool for tool in tools if tool in html_content.lower()]
        except Exception as e:
            logging.error(f"Error detecting marketing tools: {str(e)}")
            return []

    def detect_development_tech(self, html_content: str) -> List[str]:
        try:
            tech = ['react', 'angular', 'vue', 'node.js', 'python', 'java']
            return [t for t in tech if t in html_content.lower()]
        except Exception as e:
            logging.error(f"Error detecting development tech: {str(e)}")
            return []

    def get_empty_result(self, domain: str) -> Dict:
        return {
            'domain': domain,
            'success': False,
            'data': self.get_empty_data_structure()
        }

    def get_empty_data_structure(self) -> Dict:
        return {
            'company_profile': {},
            'contact_information': {'emails': [], 'phones': [], 'addresses': [], 'social_links': {}},
            'technology_stack': {},
            'business_signals': {},
            'market_position': {},
            'decision_makers': []
        }

    def get_empty_company_profile(self) -> Dict:
        return {
            'name': '',
            'industry': 'unknown',
            'company_size': 'unknown',
            'locations': [],
            'products_services': [],
            'description': '',
            'founded_year': None,
            'employee_count': None
        }